# Diamond Creek ETF Arbitrage — Backtest (fixed CSV weights)

Minimal slice of `Diamond_Creek_Backtest_v15.ipynb`: same config, universe, borrow, prices, costs, and **v15 engine**. **`PAIR_WEIGHTS`** are loaded from a CSV (no optimizer / sweeps).


## Setup


In [244]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, requests, time, ftplib, io, os
from datetime import datetime, timezone
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

TRADING_DAYS = 252
norm_sym = lambda x: str(x).strip().upper().replace(".", "-")
print("Imports OK")


Imports OK


## Configuration


## v15 Override — Clear Street borrow rates + beta>1.5 universe

This override loads borrow rates from the Clear Street output in `data/runs/<run_day>/clearstreet/bulk_rates_one_by_one.csv`, filters the backtest universe to ETFs present in that file and with `Beta > 1.5`, then applies those borrow rates for sizing/backtest.

In [245]:
# v15: use etfs2.csv rates as the exclusive borrow input source.

from pathlib import Path
import pandas as pd

etfs2_rate_paths = [
    Path(r"C:/Users/werdn/Downloads/etfs2.csv"),
    Path("data/backtest/etfs2.csv"),
    Path("../data/backtest/etfs2.csv"),
]
ETFS2_RATE_PATH = next((p for p in etfs2_rate_paths if p.exists()), None)
if ETFS2_RATE_PATH is None:
    raise FileNotFoundError(
        "Borrow-rate file not found. Checked C:/Users/werdn/Downloads/etfs2.csv and data/backtest/etfs2.csv."
    )

etfs2 = pd.read_csv(ETFS2_RATE_PATH)
etfs2.columns = [str(c).strip() for c in etfs2.columns]
tcol = next((c for c in ["Ticker", "ETF", "Symbol", "ticker", "etf", "symbol"] if c in etfs2.columns), None)
rcol = next((c for c in ["Rate", "rate", "borrow_rate_pct", "Borrow", "borrow"] if c in etfs2.columns), None)
if (tcol is None) or (rcol is None):
    raise KeyError(
        f"etfs2.csv missing required borrow columns. Need ticker + rate; found {list(etfs2.columns)}"
    )

_tmp = etfs2[[tcol, rcol]].copy()
_tmp[tcol] = _tmp[tcol].astype(str).str.upper().str.strip().str.replace(".", "-", regex=False)
_tmp[rcol] = pd.to_numeric(_tmp[rcol], errors="coerce")
_tmp = _tmp[_tmp[tcol].ne("") & _tmp[rcol].notna()].drop_duplicates(subset=[tcol], keep="last")

# Backtest convention: positive BORROW_MAP means we pay borrow.
# etfs2 convention used here: positive means rebate/credit, negative means fee/cost.
# Convert to decimal and invert sign to match backtest convention.
etf2_borrow_map_all = {
    str(t): float(-r / 100.0)
    for t, r in zip(_tmp[tcol], _tmp[rcol])
}

# Flags consumed by downstream cells.
V15_ETFS2_OVERRIDE_ACTIVE = True
V15_ETFS2_TICKERS_WITH_RATES = set(etf2_borrow_map_all.keys())
V15_BORROW_TICKERS_WITH_RATES = set(etf2_borrow_map_all.keys())
# Backward-compatible aliases for older downstream references.
V13_ETFS2_OVERRIDE_ACTIVE = V15_ETFS2_OVERRIDE_ACTIVE
V13_ETFS2_TICKERS_WITH_RATES = V15_ETFS2_TICKERS_WITH_RATES
SAMPLE_BOOK_BORROW_MAP = dict(etf2_borrow_map_all)

print(f"[V15] etfs2 borrow source: {ETFS2_RATE_PATH}")
print(f"[V15] etfs2 rows parsed: {len(_tmp)}")
print(f"[V15] Total borrow-map tickers: {len(V15_BORROW_TICKERS_WITH_RATES)}")
print("[V15] Universe rule: include ETFs present in etfs2 borrow-rate file")

[V15] etfs2 borrow source: C:\Users\werdn\Downloads\etfs2.csv
[V15] etfs2 rows parsed: 389
[V15] Total borrow-map tickers: 389
[V15] Universe rule: include ETFs present in etfs2 borrow-rate file


In [246]:
CFG = {
    "capital_usd":          10_000_000,
    "start_date":           "2023-01-01",
    "slippage_bps":         10,
    # Clear Street US equities low-touch baseline (35 mills/share).
    "clear_street_comm_per_share":  0.0035,
    "clear_street_comm_min":        0.00,
    "clear_street_comm_max_pct":    1.00,
    "fallback_borrow_rate": 0.0,
    # Clear Street financing assumptions from proposal:
    # debit = OBFR + 45 bps, cash credit / short rebate = OBFR - 40 bps,
    # negative benchmark/rate floored at 0, Actual/360 accrual.
    "margin_debit_spreads": [
        (float("inf"), 0.0045),
    ],
    "credit_spread":        -0.0040,
    "enable_short_credit_income": False,
    "financing_daycount":   360,
    "skip_ftp":             False,
    "dead_band_pct":        0.02,
    "gross_dead_band_pct":  0.05,       # full position resize if portfolio gross drifts > 5% from target
    "weight_power":         1.15,       # mild convexity to lift return without aggressive concentration
    "mix_decay_weight":     0.40,       # balanced blend between realized PnL/Gross and decay signal
    "decay_weight_power":   1.05,       # slight emphasis on higher-quality decay names
    "max_dd_penalty_weight": 0.50,      # stronger explicit penalty for high drawdown names
    "max_dd_penalty_start": "2024-01-01", # focus DD penalty on 2024+ path quality
    "max_dd_penalty_recent_weight": 0.70, # heavier weight on recent-window DD vs full-history DD
    "position_concentration_power": 1.00, # neutral concentration tilt
    "short_avail_use_pct":  0.25,       # softer cap: can target up to 25% of reported shares_available
    "max_pair_weight_cap":  0.050,      # slightly higher cap to recover some upside
    "max_underlying_weight_cap": 0.11,  # mild relaxation for CAGR while keeping concentration bounded
    "missing_shares_cap":   0.02,       # strict fallback cap when shares_available is missing
    "cap_ramp_rebals":      4,          # slower ramp into constrained names
}

LEVERAGE_RUNS = [4.75]

V7_TARGET_LEV_START = 4.5   # incumbent day-0 sizing; floor for target multiple
V7_TARGET_LEV_END = 5    # modestly higher ceiling for CAGR once post-start pairs are on book

print(f"Capital: ${CFG['capital_usd']:,}  |  Start: {CFG['start_date']}")
print(f"Leverage: {LEVERAGE_RUNS}  |  v8 target gross multiple: {V7_TARGET_LEV_START}x → {V7_TARGET_LEV_END}x")
print("Weekly dead-band hedge  |  0% beta target  |  v8 combined weights (PnL/Gross + decay + shares cap)")


Capital: $10,000,000  |  Start: 2023-01-01
Leverage: [4.75]  |  v8 target gross multiple: 4.5x → 5x
Weekly dead-band hedge  |  0% beta target  |  v8 combined weights (PnL/Gross + decay + shares cap)


## PM margin — Diamond Creek sample book

Long/short equity book (~70 US names, mainly large caps and ETFs). **Source:** `data/backtest/PM Report Diamond Creek 20260326 CV.xlsx` → *Top Level Summary*.

PM notes (narrative): regulatory-style requirement lands near **$6.5MM**; **discretionary** add-on tied to crypto proxies (BITU, BITX, ETHA, ETHT, ETHU, IBIT, SOEZ); **concentration** add-on on largest names (e.g. NVDA, TSLA); small **liquidity** add-on (SOEZ). **PM/GMV** in the file is *Effective margin requirement ÷ GMV* (~17%).

The next cell loads the spreadsheet and estimates **margin excess** if gross exposure were raised to **4.5× equity** (same equity, higher GMV), by applying the file’s **MR/GMV** ratio to the new gross (equivalent to scaling today’s effective margin **linearly with GMV** when the report is internally consistent).


## Universe — v7 (same hardcoded list as v6 from `DC_Universe_Map_Filtered.xlsx`)



In [247]:
# Hardcoded from DC_Universe_Map_Filtered.xlsx (sheet "Universe Map") — all ETF slots, |β| rounded to 3dp.
CANDIDATES = [
    ("SMU", "SMR", 1.986),   ("SMUP", "SMR", 2.058),  ("QBTX", "QBTS", 1.993),
    ("BMNU", "BMNR", 1.944), ("BMNG", "BMNR", 1.993), ("ASTX", "ASTS", 1.990),
    ("CRWG", "CRWV", 1.997), ("CWVX", "CRWV", 1.990), ("WULX", "WULF", 1.997),
    ("NEBX", "NBIS", 1.999), ("NBIL", "NBIS", 1.998), ("NBIG", "NBIS", 2.009),
    ("SMCL", "SMCI", 1.990), ("CRCA", "CRCL", 1.978), ("CRCG", "CRCL", 1.990),
    ("LABX", "ALAB", 1.994), ("BEX", "BE", 1.977),    ("CLSX", "CLSK", 1.999),
    ("GLXU", "GLXY", 2.012), ("XXRP", "XRPZ", 2.020), ("XRPT", "XRPZ", 2.018),
    ("ETHU", "ETHA", 2.001), ("ETHT", "ETHA", 1.991), ("ETU", "ETHA", 1.990),
    ("CRDU", "CRDO", 1.988), ("FIGG", "FIG", 1.996),  ("SOLT", "SOEZ", 2.014),
    ("CONL", "COIN", 1.987), ("APPX", "APP", 1.984),  ("UUUG", "UUUU", 1.995),
    ("SOXL", "SOXX", 2.960), ("ROBN", "HOOD", 2.012), ("HOOG", "HOOD", 2.007),
    ("LITX", "LITE", 1.980), ("INTW", "INTC", 1.988), ("UPSX", "UPST", 1.989),
    ("CSEX", "CLS", 1.998),  ("MUU", "MU", 1.994),    ("MULL", "MU", 1.997),
    ("ARCX", "ACHR", 1.989), ("SATG", "SATS", 2.010), ("VRTL", "VRT", 1.982),
    ("GDXU", "GDX", 3.043),  ("NUGT", "GDX", 1.981),  ("TEMT", "TEM", 1.985),
    ("RDTL", "RDDT", 1.982), ("RBLU", "RBLX", 2.019), ("AMDL", "AMD", 1.997),
    ("AMUU", "AMD", 2.008),  ("AMDG", "AMD", 2.003),  ("TSLL", "TSLA", 1.993),
    ("TSLT", "TSLA", 1.993), ("TSLR", "TSLA", 1.999), ("TSLG", "TSLA", 1.991),
    ("BITX", "IBIT", 2.005), ("BITU", "IBIT", 1.992), ("BTCL", "IBIT", 2.007),
    ("YINN", "FXI", 2.974),  ("MVLL", "MRVL", 1.992), ("URAA", "URA", 1.933),
    ("SNOU", "SNOW", 2.024), ("BULG", "BULL", 1.985), ("PLTU", "PLTR", 1.992),
    ("PTIR", "PLTR", 1.997), ("PLTG", "PLTR", 1.996), ("BABX", "BABA", 2.006),
    ("NVDL", "NVDA", 1.988), ("NVDU", "NVDA", 1.986), ("NVDX", "NVDA", 1.997),
    ("NVDG", "NVDA", 1.969), ("LRCU", "LRCX", 2.045), ("ARMG", "ARM", 1.997),
    ("TSMU", "TSM", 1.990),  ("TSMG", "TSM", 1.979),  ("GEVX", "GEV", 1.993),
    ("KTUP", "KTOS", 2.026), ("AVL", "AVGO", 1.987),  ("AVGG", "AVGO", 1.999),
    ("AVGU", "AVGO", 1.992), ("UBRL", "UBER", 1.995), ("CRMG", "CRM", 1.984),
    ("UNHG", "UNH", 1.996),  ("MEXX", "EWW", 2.958),  ("JNUG", "GDXJ", 1.974),
    ("NFLU", "NFLX", 1.990), ("CWEB", "KWEB", 1.987), ("CEGX", "CEG", 1.979),
    ("GGLL", "GOOGL", 1.992),("GOOX", "GOOGL", 1.984),("KORU", "EWY", 2.931),
    ("TERG", "TER", 1.996),  ("DLLL", "DELL", 1.986), ("METU", "META", 1.999),
    ("FBL", "META", 1.997),  ("AMZU", "AMZN", 1.987), ("AMZZ", "AMZN", 1.992),
    ("PALU", "PANW", 2.006), ("ASMG", "ASML", 1.984), ("WDCX", "WDC", 2.055),
    ("CRWL", "CRWD", 1.997), ("ELIL", "LLY", 1.991),  ("PYPG", "PYPL", 1.995),
    ("BOEU", "BA", 1.987),   ("MSFU", "MSFT", 1.991), ("MSFL", "MSFT", 1.992),
    ("MSFX", "MSFT", 2.007), ("CHAU", "ASHR", 1.967), ("NOWL", "NOW", 1.996),
    ("ERX", "XLE", 1.988),   ("GUSH", "XOP", 2.011),  ("SHPU", "SHOP", 2.012),
    ("XOMX", "XOM", 1.969),  ("ORCU", "ORCL", 2.006), ("ADBG", "ADBE", 1.992),
    ("AAPU", "AAPL", 1.984), ("AAPB", "AAPL", 1.991), ("AAPX", "AAPL", 1.995),
    ("TARK", "ARKK", 1.978), ("LABU", "XBI", 2.988),  ("CSCL", "CSCO", 2.027),
    ("LMTL", "LMT", 1.984),  ("COTG", "COST", 1.987),
]

# Bucket 3 pairs (inverse ETF + underlying), handled with short-inverse + short-underlying leg construction.
BUCKET3_CANDIDATES = [
    ("APLZ", "APLD"),
    ("CLSZ", "CLSK"),
    ("IREZ", "IREN"),
    ("BEZ", "BE"),
    ("NBIZ", "NBIS"),
    ("SMZ", "SMR"),
    ("MSTZ", "MSTR"),
    ("TSLQ", "TSLA"),
    ("NVDQ", "NVDA"),
    ("SOXS", "SOXX"),
]
BUCKET3_ETFS = {e for e, _ in BUCKET3_CANDIDATES}
BUCKET3_UNDERLYINGS = {u for _, u in BUCKET3_CANDIDATES}

BASE_CANDIDATE_KEYS = {(e, u) for e, u, _ in CANDIDATES}

# Add all remaining screened pairs not already in the base map.
screen_paths = [Path("../data/etf_screened_today.csv"), Path("data/etf_screened_today.csv")]
_screen_path = next((p for p in screen_paths if p.exists()), None)
_added = 0
if _screen_path is not None:
    _sc = pd.read_csv(_screen_path)
    _sc.columns = [str(c).strip() for c in _sc.columns]
    req = {"ETF", "Underlying", "Beta"}
    if req.issubset(_sc.columns):
        _sc["ETF"] = _sc["ETF"].astype(str).str.upper().str.strip()
        _sc["Underlying"] = _sc["Underlying"].astype(str).str.upper().str.strip()
        _sc["Beta"] = pd.to_numeric(_sc["Beta"], errors="coerce")

        for r in _sc.itertuples(index=False):
            e = r.ETF
            u = r.Underlying
            b = float(r.Beta) if pd.notna(r.Beta) else np.nan
            if not e or not u or not np.isfinite(b):
                continue
            key = (e, u)
            if key in BASE_CANDIDATE_KEYS:
                continue
            CANDIDATES.append((e, u, round(abs(b), 3)))
            BASE_CANDIDATE_KEYS.add(key)
            _added += 1

EXTRA_ETFS = {e for e, u, _ in CANDIDATES if (e, u) not in {(x, y) for x, y, _ in [
    ("SMU", "SMR", 1),   ("SMUP", "SMR", 1),  ("QBTX", "QBTS", 1),
    ("BMNU", "BMNR", 1), ("BMNG", "BMNR", 1), ("ASTX", "ASTS", 1),
    ("CRWG", "CRWV", 1), ("CWVX", "CRWV", 1), ("WULX", "WULF", 1),
    ("NEBX", "NBIS", 1), ("NBIL", "NBIS", 1), ("NBIG", "NBIS", 1),
    ("SMCL", "SMCI", 1), ("CRCA", "CRCL", 1), ("CRCG", "CRCL", 1),
    ("LABX", "ALAB", 1), ("BEX", "BE", 1),    ("CLSX", "CLSK", 1),
    ("GLXU", "GLXY", 1), ("XXRP", "XRPZ", 1), ("XRPT", "XRPZ", 1),
    ("ETHU", "ETHA", 1), ("ETHT", "ETHA", 1), ("ETU", "ETHA", 1),
    ("CRDU", "CRDO", 1), ("FIGG", "FIG", 1),  ("SOLT", "SOEZ", 1),
    ("CONL", "COIN", 1), ("APPX", "APP", 1),  ("UUUG", "UUUU", 1),
    ("SOXL", "SOXX", 1), ("ROBN", "HOOD", 1), ("HOOG", "HOOD", 1),
    ("LITX", "LITE", 1), ("INTW", "INTC", 1), ("UPSX", "UPST", 1),
    ("CSEX", "CLS", 1),  ("MUU", "MU", 1),    ("MULL", "MU", 1),
    ("ARCX", "ACHR", 1), ("SATG", "SATS", 1), ("VRTL", "VRT", 1),
    ("GDXU", "GDX", 1),  ("NUGT", "GDX", 1),  ("TEMT", "TEM", 1),
    ("RDTL", "RDDT", 1), ("RBLU", "RBLX", 1), ("AMDL", "AMD", 1),
    ("AMUU", "AMD", 1),  ("AMDG", "AMD", 1),  ("TSLL", "TSLA", 1),
    ("TSLT", "TSLA", 1), ("TSLR", "TSLA", 1), ("TSLG", "TSLA", 1),
    ("BITX", "IBIT", 1), ("BITU", "IBIT", 1), ("BTCL", "IBIT", 1),
    ("YINN", "FXI", 1),  ("MVLL", "MRVL", 1), ("URAA", "URA", 1),
    ("SNOU", "SNOW", 1), ("BULG", "BULL", 1), ("PLTU", "PLTR", 1),
    ("PTIR", "PLTR", 1), ("PLTG", "PLTR", 1), ("BABX", "BABA", 1),
    ("NVDL", "NVDA", 1), ("NVDU", "NVDA", 1), ("NVDX", "NVDA", 1),
    ("NVDG", "NVDA", 1), ("LRCU", "LRCX", 1), ("ARMG", "ARM", 1),
    ("TSMU", "TSM", 1),  ("TSMG", "TSM", 1),  ("GEVX", "GEV", 1),
    ("KTUP", "KTOS", 1), ("AVL", "AVGO", 1),  ("AVGG", "AVGO", 1),
    ("AVGU", "AVGO", 1), ("UBRL", "UBER", 1), ("CRMG", "CRM", 1),
    ("UNHG", "UNH", 1),  ("MEXX", "EWW", 1),  ("JNUG", "GDXJ", 1),
    ("NFLU", "NFLX", 1), ("CWEB", "KWEB", 1), ("CEGX", "CEG", 1),
    ("GGLL", "GOOGL", 1),("GOOX", "GOOGL", 1),("KORU", "EWY", 1),
    ("TERG", "TER", 1),  ("DLLL", "DELL", 1), ("METU", "META", 1),
    ("FBL", "META", 1),  ("AMZU", "AMZN", 1), ("AMZZ", "AMZN", 1),
    ("PALU", "PANW", 1), ("ASMG", "ASML", 1), ("WDCX", "WDC", 1),
    ("CRWL", "CRWD", 1), ("ELIL", "LLY", 1),  ("PYPG", "PYPL", 1),
    ("BOEU", "BA", 1),   ("MSFU", "MSFT", 1), ("MSFL", "MSFT", 1),
    ("MSFX", "MSFT", 1), ("CHAU", "ASHR", 1), ("NOWL", "NOW", 1),
    ("ERX", "XLE", 1),   ("GUSH", "XOP", 1),  ("SHPU", "SHOP", 1),
    ("XOMX", "XOM", 1),  ("ORCU", "ORCL", 1), ("ADBG", "ADBE", 1),
    ("AAPU", "AAPL", 1), ("AAPB", "AAPL", 1), ("AAPX", "AAPL", 1),
    ("TARK", "ARKK", 1), ("LABU", "XBI", 1),  ("CSCL", "CSCO", 1),
    ("LMTL", "LMT", 1),  ("COTG", "COST", 1),
]}}

# Rebuild additions with explicit Beta filter: include only screened ETFs with Beta > 1.5.
_base_keys = {(e, u) for e, u, _ in CANDIDATES if e not in EXTRA_ETFS}
base_candidates = [t for t in CANDIDATES if (t[0], t[1]) in _base_keys]

extra_candidates = []
EXTRA_ETFS = set()
if _screen_path is not None and req.issubset(_sc.columns):
    for r in _sc.itertuples(index=False):
        e = str(r.ETF).upper().strip()
        u = str(r.Underlying).upper().strip()
        b = float(r.Beta) if pd.notna(r.Beta) else np.nan
        if not e or not u or not np.isfinite(b):
            continue
        if b <= 1.5:
            continue
        key = (e, u)
        if key in _base_keys:
            continue
        extra_candidates.append((e, u, round(b, 3)))
        EXTRA_ETFS.add(e)

# Deduplicate while preserving order.
_seen = set()
CANDIDATES = []
for e, u, b in base_candidates + extra_candidates:
    k = (e, u)
    if k in _seen:
        continue
    _seen.add(k)
    CANDIDATES.append((e, u, b))

# Load whitelist ETF universe from strategy_config.yml (keep even if beta < cutoff).
WHITELIST_ETFS = set()
cfg_candidates = [Path("config/strategy_config.yml"), Path("../config/strategy_config.yml")]
cfg_path = next((p for p in cfg_candidates if p.exists()), None)
if cfg_path is not None:
    try:
        import yaml
        with open(cfg_path, "r", encoding="utf-8") as f:
            _cfg_obj = yaml.safe_load(f) or {}
        _wl = (((_cfg_obj.get("portfolio", {}) or {}).get("sleeves", {}) or {}).get("whitelist_stock", {}) or {}).get("universe", {}).get("etfs", [])
        WHITELIST_ETFS = {
            str(x).upper().strip().replace(".", "-")
            for x in (_wl or [])
            if str(x).strip()
        }
    except Exception:
        WHITELIST_ETFS = set()

# Add whitelist pairs even when beta <= cutoff.
_base_etf_to_und = {}
for _e0, _u0, _b0 in base_candidates:
    _base_etf_to_und.setdefault(str(_e0).upper().strip().replace(".", "-"), str(_u0).upper().strip())

_screen_etf_to_und = {}
if _screen_path is not None and req.issubset(_sc.columns):
    _tmp_sc = _sc[["ETF", "Underlying"]].dropna().copy()
    _tmp_sc["ETF"] = _tmp_sc["ETF"].astype(str).str.upper().str.strip().str.replace(".", "-", regex=False)
    _tmp_sc["Underlying"] = _tmp_sc["Underlying"].astype(str).str.upper().str.strip()
    _tmp_sc = _tmp_sc[(_tmp_sc["ETF"] != "") & (_tmp_sc["Underlying"] != "")]
    _tmp_sc = _tmp_sc.drop_duplicates(subset=["ETF"], keep="first")
    _screen_etf_to_und = dict(zip(_tmp_sc["ETF"], _tmp_sc["Underlying"]))

_added_wl = 0
for _e in sorted(WHITELIST_ETFS):
    _u = _screen_etf_to_und.get(_e, _base_etf_to_und.get(_e, ""))
    if not _u:
        continue
    _k = (_e, _u)
    if _k in _seen:
        continue
    CANDIDATES.append((_e, _u, 1.0))
    _seen.add(_k)
    _added_wl += 1
if _added_wl > 0:
    print(f"[V15] Added whitelist pairs: {_added_wl}")

# Add bucket 3 pairs even when beta <= cutoff.
_added_b3 = 0
for _e, _u in BUCKET3_CANDIDATES:
    _e = str(_e).upper().strip().replace(".", "-")
    _u = str(_u).upper().strip()
    _k = (_e, _u)
    if _k in _seen:
        continue
    _b3_beta = 1.0
    CANDIDATES.append((_e, _u, round(float(_b3_beta), 3)))
    _seen.add(_k)
    _added_b3 += 1
if _added_b3 > 0:
    print(f"[V15] Added bucket3 pairs: {_added_b3}")

# Global beta filter for the entire v9 backtest universe.
MIN_BETA_ABS = 1.5
pair_beta_from_screen = {}
etf_beta_from_screen = {}
if _screen_path is not None and req.issubset(_sc.columns):
    for r in _sc.itertuples(index=False):
        e = str(r.ETF).upper().strip()
        u = str(r.Underlying).upper().strip()
        b = float(r.Beta) if pd.notna(r.Beta) else np.nan
        if not e or not u or not np.isfinite(b):
            continue
        pair_beta_from_screen[(e, u)] = abs(b)
        etf_beta_from_screen[e] = abs(b)

_before = len(CANDIDATES)
_filtered = []
for e, u, b in CANDIDATES:
    b_abs = pair_beta_from_screen.get((e, u), etf_beta_from_screen.get(e, abs(float(b))))
    _etf_norm = str(e).upper().strip().replace(".", "-")
    _is_whitelist = _etf_norm in WHITELIST_ETFS
    _is_bucket3 = _etf_norm in BUCKET3_ETFS
    if (not _is_whitelist) and (not _is_bucket3) and (not np.isfinite(b_abs) or b_abs <= MIN_BETA_ABS):
        continue
    _filtered.append((e, u, round(float(b_abs), 3)))
CANDIDATES = _filtered

# v15 clearstreet override: keep ETFs present in rates file; preserve whitelist + bucket3 pairs.
if globals().get("V15_ETFS2_OVERRIDE_ACTIVE", globals().get("V13_ETFS2_OVERRIDE_ACTIVE", False)):
    _etfs2_ok = {
        str(x).upper().strip().replace(".", "-")
        for x in globals().get("V15_ETFS2_TICKERS_WITH_RATES", globals().get("V13_ETFS2_TICKERS_WITH_RATES", set()))
    }
    if _etfs2_ok:
        _before_etfs2 = len(CANDIDATES)
        CANDIDATES = [
            (e, u, b)
            for e, u, b in CANDIDATES
            if (
                str(e).upper().strip().replace(".", "-") in _etfs2_ok
                or str(e).upper().strip().replace(".", "-") in WHITELIST_ETFS
                or str(e).upper().strip().replace(".", "-") in BUCKET3_ETFS
            )
        ]
        print(f"[V15] Applied clearstreet ticker/rate filter: {_before_etfs2} -> {len(CANDIDATES)} pairs")

print(
    f"Total candidate pairs: {len(CANDIDATES)} | Added from screener (positive Beta>{MIN_BETA_ABS}): {len(extra_candidates)} "
    f"| Dropped by global beta filter: {_before - len(CANDIDATES)}"
)
print(f"Additional ETF names (capacity-sized only): {len(EXTRA_ETFS)}")
print(f"[V15] Whitelist ETFs loaded from config: {len(WHITELIST_ETFS)}")
print(f"[V15] Bucket3 ETFs configured: {len(BUCKET3_ETFS)}")


[V15] Added whitelist pairs: 18
[V15] Added bucket3 pairs: 10
[V15] Applied clearstreet ticker/rate filter: 320 -> 279 pairs
Total candidate pairs: 279 | Added from screener (positive Beta>1.5): 170 | Dropped by global beta filter: 41
Additional ETF names (capacity-sized only): 170
[V15] Whitelist ETFs loaded from config: 18
[V15] Bucket3 ETFs configured: 10


In [248]:
# ---- Restrict universe to Clear Street sample portfolio pairs ----
sample_paths = [
    Path(r"C:/Users/werdn/Downloads/Copy of clear_street_sample_portfolio (1) (002)1.xlsx"),
    Path("data/backtest/clear_street_sample_portfolio.csv"),
    Path("../data/backtest/clear_street_sample_portfolio.csv"),
]
USE_SAMPLE_BOOK_ONLY = bool(globals().get("USE_SAMPLE_BOOK_ONLY", False))
if globals().get("V15_ETFS2_OVERRIDE_ACTIVE", globals().get("V13_ETFS2_OVERRIDE_ACTIVE", False)):
    USE_SAMPLE_BOOK_ONLY = False
    print("[SAMPLE] v15 clearstreet override active: sample-portfolio restriction disabled.")

if USE_SAMPLE_BOOK_ONLY:
    SAMPLE_BOOK_PATH = next((p for p in sample_paths if p.exists()), None)
else:
    SAMPLE_BOOK_PATH = None
    print("[SAMPLE] Using full equal-weight candidate universe for sizing (not sample-only).")

if SAMPLE_BOOK_PATH is None:
    print("[SAMPLE] Sample portfolio file not found; keeping existing CANDIDATES universe.")
    sample_df = pd.DataFrame(columns=["Pair"])
elif SAMPLE_BOOK_PATH.suffix.lower() == ".csv":
    sample_df = pd.read_csv(SAMPLE_BOOK_PATH)
else:
    sample_df = pd.read_excel(SAMPLE_BOOK_PATH)

sample_df.columns = [str(c).strip() for c in sample_df.columns]
if "Pair" not in sample_df.columns:
    print("[SAMPLE] Missing Pair column; keeping existing CANDIDATES universe.")
    sample_df = pd.DataFrame(columns=["Pair"])

sample_df["Pair"] = sample_df["Pair"].astype(str).str.upper().str.strip()
sample_df = sample_df[sample_df["Pair"].str.contains("/")].copy()

# Existing beta fallback map from the hardcoded universe loaded above.
_base_beta = {(e, u): abs(float(b)) for e, u, b in CANDIDATES}

# Optional beta fallback from current screener.
screen_beta = {}
for _p in [Path("data/etf_screened_today.csv"), Path("../data/etf_screened_today.csv")]:
    if _p.exists():
        _s = pd.read_csv(_p)
        _s.columns = [str(c).strip() for c in _s.columns]
        if {"ETF", "Underlying", "Beta"}.issubset(_s.columns):
            _s["ETF"] = _s["ETF"].astype(str).str.upper().str.strip()
            _s["Underlying"] = _s["Underlying"].astype(str).str.upper().str.strip()
            _s["Beta"] = pd.to_numeric(_s["Beta"], errors="coerce")
            for _r in _s.itertuples(index=False):
                if pd.notna(_r.Beta):
                    screen_beta[(_r.ETF, _r.Underlying)] = abs(float(_r.Beta))
        break

sample_pairs = []
for pair in sample_df["Pair"].dropna().astype(str).tolist():
    und, etf = [x.strip().upper() for x in pair.split("/", 1)]
    if not und or not etf:
        continue
    sample_pairs.append((etf, und))

sample_pairs = list(dict.fromkeys(sample_pairs))

# Build sample-only candidate list with robust beta fallback.
sample_candidates = []
missing_beta = []
for etf, und in sample_pairs:
    b = _base_beta.get((etf, und), screen_beta.get((etf, und), np.nan))
    if not np.isfinite(b):
        b = 2.0  # conservative fallback for leveraged ETF hedging
        missing_beta.append((etf, und))
    sample_candidates.append((etf, und, round(abs(float(b)), 3)))

if sample_candidates:
    CANDIDATES = sample_candidates
else:
    if USE_SAMPLE_BOOK_ONLY:
        print("[SAMPLE] No valid sample pairs found; keeping existing CANDIDATES universe.")

# Borrow-rate column in sample sheet is currently Unnamed: 6.
# Convention requested by user: positive means we GET PAID to short.
# Backtest BORROW_MAP convention: positive = we PAY borrow. So we invert sign.
if globals().get("V15_ETFS2_OVERRIDE_ACTIVE", globals().get("V13_ETFS2_OVERRIDE_ACTIVE", False)):
    SAMPLE_BOOK_BORROW_MAP = dict(globals().get("SAMPLE_BOOK_BORROW_MAP", {}))
else:
    borrow_col = "Unnamed: 6" if "Unnamed: 6" in sample_df.columns else None
    if borrow_col is None:
        SAMPLE_BOOK_BORROW_MAP = {}
    else:
        tmp = sample_df[["Pair", borrow_col]].copy()
        tmp[borrow_col] = pd.to_numeric(tmp[borrow_col], errors="coerce")
        tmp = tmp[tmp[borrow_col].notna()].copy()
        bmap = {}
        for _, r in tmp.iterrows():
            und, etf = [x.strip().upper() for x in str(r["Pair"]).split("/", 1)]
            raw_pct_points = float(r[borrow_col])
            bmap[etf] = -(raw_pct_points / 100.0)
        SAMPLE_BOOK_BORROW_MAP = bmap

if USE_SAMPLE_BOOK_ONLY:
    print(f"Sample-book universe active: {len(CANDIDATES)} pairs")
    print(f"Sample-book borrow rows: {len(SAMPLE_BOOK_BORROW_MAP)} ETFs (sign-inverted: +sheet => short credit)")
else:
    print(f"Full-universe sizing mode active: {len(CANDIDATES)} pairs")
    print(f"Borrow rows available in map: {len(SAMPLE_BOOK_BORROW_MAP)} ETFs")
if missing_beta:
    print(f"Pairs with fallback beta=2.0: {len(missing_beta)}")


[SAMPLE] v15 clearstreet override active: sample-portfolio restriction disabled.
[SAMPLE] Using full equal-weight candidate universe for sizing (not sample-only).
[SAMPLE] Sample portfolio file not found; keeping existing CANDIDATES universe.
Full-universe sizing mode active: 279 pairs
Borrow rows available in map: 389 ETFs


## Data — Borrow Rates, Prices, Pair Selection


In [249]:
# ---- Borrow rates (historical average first, FTP/cache fallback) ----
all_etf_syms = [e for e, _, _ in CANDIDATES]
BORROW_CACHE = Path("data/borrow_cache.csv")

# Resolve runs path robustly across notebook cwd variants.
runs_candidates = [
    Path("data/runs"),
    Path("../data/runs"),
    Path.cwd() / "data/runs",
    Path.cwd() / "../data/runs",
    Path.cwd().parent / "data/runs",
]
RUNS_ROOT = next((p.resolve() for p in runs_candidates if p.exists()), None)
if RUNS_ROOT is None:
    RUNS_ROOT = Path("data/runs")

# Build consolidated historical panel (borrow + shares) from run snapshots.
def _trimmed_mean_drop2(s: pd.Series) -> float:
    x = pd.to_numeric(s, errors="coerce").dropna().sort_values()
    if len(x) <= 4:
        return float(x.mean()) if len(x) else np.nan
    return float(x.iloc[2:-2].mean())


def _norm_col(c: str) -> str:
    return "".join(ch for ch in str(c).lower().strip() if ch.isalnum())


def _pick_col(cols: list[str], exact_norm: list[str], fuzzy_contains: list[str] | None = None) -> str | None:
    norm_to_orig = {_norm_col(c): c for c in cols}
    for k in exact_norm:
        if k in norm_to_orig:
            return norm_to_orig[k]
    if fuzzy_contains:
        for c in cols:
            nc = _norm_col(c)
            if all(tok in nc for tok in fuzzy_contains):
                return c
    return None


# Prefer etf_screened_today for a date, but still use proposed_trades to fill missing fields.
def _historical_snapshot_files(root: Path) -> list[tuple[pd.Timestamp, Path, int]]:
    out = []
    for d in sorted(root.glob("*")):
        if not d.is_dir():
            continue
        ds = pd.to_datetime(d.name, errors="coerce")
        if pd.isna(ds):
            continue
        f_screen = d / "etf_screened_today.csv"
        f_trades = d / "proposed_trades.csv"
        if f_screen.exists():
            out.append((ds, f_screen, 0))
        if f_trades.exists():
            out.append((ds, f_trades, 1))

    # Include current latest screener/proposed files if present.
    extra = [
        (Path("data/etf_screened_today.csv"), 2),
        (Path("../data/etf_screened_today.csv"), 2),
        (Path("data/proposed_trades.csv"), 3),
        (Path("../data/proposed_trades.csv"), 3),
    ]
    for p, rank in extra:
        if p.exists():
            dt = pd.to_datetime(p.stat().st_mtime, unit="s", errors="coerce")
            out.append((dt, p, rank))
    return out


def _extract_snapshot_rows(ds: pd.Timestamp, fp: Path, src_rank: int) -> pd.DataFrame:
    try:
        d = pd.read_csv(fp)
    except Exception:
        return pd.DataFrame(columns=["date", "etf", "borrow_rate", "shares_available", "src_rank", "src_file"])
    if d.empty:
        return pd.DataFrame(columns=["date", "etf", "borrow_rate", "shares_available", "src_rank", "src_file"])

    cols = list(d.columns)
    etf_col = _pick_col(cols, ["etf", "ticker", "symbol", "sym"], None)
    if etf_col is None:
        return pd.DataFrame(columns=["date", "etf", "borrow_rate", "shares_available", "src_rank", "src_file"])

    borrow_col = _pick_col(
        cols,
        [
            "borrownetannual",
            "borrowcurrent",
            "borrowfeeannual",
            "borrowrate",
            "feeborrow",
            "borrowfee",
            "shortborrowrate",
            "shortfee",
        ],
        ["borrow"],
    )

    shares_col = _pick_col(
        cols,
        [
            "sharesavailable",
            "sharesavail",
            "availableshares",
            "sharesavailableforshort",
            "shortsharesavailable",
        ],
        ["share", "avail"],
    )

    out = pd.DataFrame()
    out["date"] = [ds] * len(d)
    out["etf"] = d[etf_col].astype(str).str.upper().str.strip()
    out["borrow_rate"] = pd.to_numeric(d[borrow_col], errors="coerce") if borrow_col is not None else np.nan
    out["shares_available"] = pd.to_numeric(d[shares_col], errors="coerce") if shares_col is not None else np.nan
    out["src_rank"] = src_rank
    out["src_file"] = str(fp)
    out = out[out["etf"].ne("")].copy()
    return out[["date", "etf", "borrow_rate", "shares_available", "src_rank", "src_file"]]


hist_raw = []
for ds, fp, src_rank in _historical_snapshot_files(RUNS_ROOT):
    rows = _extract_snapshot_rows(ds, fp, src_rank)
    if not rows.empty:
        hist_raw.append(rows)

if hist_raw:
    hist_raw_df = pd.concat(hist_raw, ignore_index=True)

    def _first_valid(s: pd.Series) -> float:
        x = pd.to_numeric(s, errors="coerce").dropna()
        return float(x.iloc[0]) if len(x) else np.nan

    # Per date/ticker, keep first valid field with file priority: screen -> proposed_trades -> local latest files.
    hist_raw_df = hist_raw_df.sort_values(["date", "etf", "src_rank"]).reset_index(drop=True)
    HIST_SNAPSHOT_PANEL = (
        hist_raw_df.groupby(["date", "etf"], as_index=False)
        .agg(
            borrow_rate=("borrow_rate", _first_valid),
            shares_available=("shares_available", _first_valid),
        )
        .sort_values(["date", "etf"])
        .reset_index(drop=True)
    )

    # Maps consumed by older downstream cells.
    hist_avg_map = HIST_SNAPSHOT_PANEL.groupby("etf")["borrow_rate"].apply(_trimmed_mean_drop2).to_dict()
    HIST_BORROW_AVG_MAP = hist_avg_map
    HIST_SHARES_AVG_MAP = HIST_SNAPSHOT_PANEL.groupby("etf")["shares_available"].mean().to_dict()

    hist_covered = sum(1 for e in all_etf_syms if e in hist_avg_map and pd.notna(hist_avg_map[e]))
    shares_covered = sum(1 for e in all_etf_syms if e in HIST_SHARES_AVG_MAP and pd.notna(HIST_SHARES_AVG_MAP[e]))
    print(
        f"[HIST] files={hist_raw_df['date'].nunique()} dates | borrow tickers={hist_covered}/{len(all_etf_syms)} | "
        f"shares tickers={shares_covered}/{len(all_etf_syms)}"
    )
else:
    HIST_SNAPSHOT_PANEL = pd.DataFrame(columns=["date", "etf", "borrow_rate", "shares_available"])
    HIST_BORROW_AVG_MAP = {}
    HIST_SHARES_AVG_MAP = {}
    hist_avg_map = {}
    print("[HIST] no usable run files found")

# Build Clear Street current map (no IBKR FTP in this notebook).
cs_map = {}
cs_shares_map = {}
cs_cov_path = Path("data/borrow_coverage_etf_screened_today.csv")
if not cs_cov_path.exists():
    cs_cov_path = Path("../data/borrow_coverage_etf_screened_today.csv")

if cs_cov_path.exists():
    cs_cov = pd.read_csv(cs_cov_path)
    if "symbol" in cs_cov.columns:
        cs_cov["symbol"] = cs_cov["symbol"].astype(str).str.upper().str.strip()
        if "borrow_fee_annual_final" in cs_cov.columns:
            v = pd.to_numeric(cs_cov["borrow_fee_annual_final"], errors="coerce")
            cs_map = {s: float(x) for s, x in zip(cs_cov["symbol"], v) if pd.notna(x)}
        if "shares_available_final" in cs_cov.columns:
            s = pd.to_numeric(cs_cov["shares_available_final"], errors="coerce")
            cs_shares_map = {sym: float(x) for sym, x in zip(cs_cov["symbol"], s) if pd.notna(x)}

    cs_cov_borrow = sum(1 for e in all_etf_syms if e in cs_map)
    cs_cov_shares = sum(1 for e in all_etf_syms if e in cs_shares_map)
    print(f"[CS] current borrow available for {cs_cov_borrow}/{len(all_etf_syms)} tickers")
    print(f"[CS] current shares available for {cs_cov_shares}/{len(all_etf_syms)} tickers")
else:
    print("[CS] borrow_coverage_etf_screened_today.csv not found; run the top Clear Street cell first.")

# Prefer live Clear Street shares where available.
if cs_shares_map:
    HIST_SHARES_AVG_MAP = {**HIST_SHARES_AVG_MAP, **cs_shares_map}

# Final borrow map used by backtest.
# For v10 sample-book mode, use ONLY spreadsheet net borrow (no historical blending).
if "SAMPLE_BOOK_BORROW_MAP" in globals() and isinstance(SAMPLE_BOOK_BORROW_MAP, dict) and SAMPLE_BOOK_BORROW_MAP:
    BORROW_MAP = {}
    hit = 0
    miss = 0
    sample_map = {str(k).upper().strip(): float(v) for k, v in SAMPLE_BOOK_BORROW_MAP.items()}
    for e in all_etf_syms:
        ee = str(e).upper().strip()
        if ee in sample_map and pd.notna(sample_map[ee]):
            BORROW_MAP[ee] = float(sample_map[ee])
            hit += 1
        else:
            BORROW_MAP[ee] = float(CFG["fallback_borrow_rate"])
            miss += 1
    print(f"[BORROW] v10 sample-book mode: spreadsheet-only borrow for {hit}/{len(all_etf_syms)} ETFs; fallback for {miss}")
else:
    # Fallback behavior if sample-book borrow is unavailable.
    BORROW_MAP = {}
    src_hist = src_cs = src_flat = 0
    for e in all_etf_syms:
        if e in hist_avg_map and pd.notna(hist_avg_map[e]):
            BORROW_MAP[e] = float(hist_avg_map[e])
            src_hist += 1
        elif e in cs_map and pd.notna(cs_map[e]):
            BORROW_MAP[e] = float(cs_map[e])
            src_cs += 1
        else:
            BORROW_MAP[e] = CFG["fallback_borrow_rate"]
            src_flat += 1
    print(f"[BORROW] source mix -> hist_avg: {src_hist}, clearstreet: {src_cs}, flat: {src_flat}")


[HIST] files=2 dates | borrow tickers=247/279 | shares tickers=279/279
[CS] current borrow available for 1/279 tickers
[CS] current shares available for 1/279 tickers
[BORROW] v10 sample-book mode: spreadsheet-only borrow for 275/279 ETFs; fallback for 4


In [250]:
# ---- Total-return prices ----
import yfinance as yf

# Manual split overrides for outlier tickers/dates (applied in raw-close path).
# Example: reverse split 1-for-10 => factor 0.1 on split date.
MANUAL_SPLIT_OVERRIDES = {
    "SMUP": {
        # For this feed representation, 10.0 neutralizes the reverse-split discontinuity.
        "2026-01-26": 10.0,
    },
    "EOSU": {
        # 1-for-25 reverse split (effective 2026-04-08): use feed-space factor.
        "2026-04-08": 25.0,
    },
}

def download_total_return(ticker, period="max"):
    try:
        tk = yf.Ticker(ticker)
        df = tk.history(period=period, auto_adjust=False, actions=True)
        if df.empty or "Close" not in df.columns:
            return None
        df = df.sort_index()

        tkr = str(ticker).upper()
        manual_splits = globals().get("MANUAL_SPLIT_OVERRIDES", {})
        force_raw_path = tkr in manual_splits

        # Prefer Yahoo adjusted close unless we explicitly need manual split overrides.
        if (not force_raw_path) and ("Adj Close" in df.columns) and (df["Adj Close"].notna().sum() > 20):
            tr = df["Adj Close"].astype(float)
            tr.index = tr.index.tz_localize(None)
            return tr.replace([np.inf, -np.inf], np.nan).dropna()

        close = df["Close"].astype(float)
        divs = (
            df["Dividends"].fillna(0).astype(float)
            if "Dividends" in df.columns
            else pd.Series(0.0, index=close.index)
        )
        splits = (
            df["Stock Splits"].replace(0, 1).fillna(1).astype(float)
            if "Stock Splits" in df.columns
            else pd.Series(1.0, index=close.index)
        )

        # Manual split overrides for outlier names/dates.
        # Apply on exact date, or nearest next trading day within 3 calendar days.
        if tkr in manual_splits:
            applied = []
            for ds, factor in manual_splits[tkr].items():
                ts = pd.Timestamp(ds)
                apply_ts = None
                if ts in splits.index:
                    apply_ts = ts
                else:
                    nxt = splits.index[splits.index >= ts]
                    if len(nxt) > 0 and (nxt[0] - ts).days <= 3:
                        apply_ts = nxt[0]

                if apply_ts is None:
                    print(f"[split-override] {tkr} {ts.date()} not applied (date not found)")
                    continue

                f = float(factor)
                old_factor = float(splits.loc[apply_ts])

                # Back-adjust all history before split day to post-split basis.
                # Here, `factor` is interpreted as the pre-split back-adjust multiplier.
                close.loc[close.index < apply_ts] = close.loc[close.index < apply_ts] * f
                divs.loc[divs.index < apply_ts] = divs.loc[divs.index < apply_ts] * f

                # Neutralize split jump on event day because history is already back-adjusted.
                splits.loc[apply_ts] = 1.0

                applied.append((ts, apply_ts, old_factor, f))

            if applied:
                for ts, apply_ts, old_factor, f in applied:
                    print(
                        f"[split-override] {tkr} requested {ts.date()} applied {apply_ts.date()} "
                        f"back-adjust x{f:g}, split-factor {old_factor:g} -> 1 (event-day neutralized)"
                    )
            else:
                print(f"[split-override] {tkr} had no applied overrides")

        # Daily total-return factor with split and dividend handling.
        gross = ((close * splits) + divs) / close.shift(1)
        gross = gross.replace([np.inf, -np.inf], np.nan).fillna(1.0)
        tr = close.iloc[0] * gross.cumprod()
        tr.index = tr.index.tz_localize(None)
        return tr.replace([np.inf, -np.inf], np.nan).dropna()
    except:
        return None

ALL_TICKERS = sorted(set(e for e, _, _ in CANDIDATES) | set(u for _, u, _ in CANDIDATES) | {"SPY"})
t0 = time.time()
print(f"Downloading {len(ALL_TICKERS)} tickers...")
PRICES = {}
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = {ex.submit(download_total_return, t): t for t in ALL_TICKERS}
    done = 0
    for f in as_completed(futs):
        done += 1
        t = futs[f]
        s = f.result()
        if s is not None and len(s) > 20:
            PRICES[t] = s
        if done % 20 == 0:
            print(f"  {done}/{len(ALL_TICKERS)} [{time.time()-t0:.0f}s]")
print(f"Got {len(PRICES)}/{len(ALL_TICKERS)} [{time.time()-t0:.1f}s]")
if globals().get("MANUAL_SPLIT_OVERRIDES"):
    print("Manual split overrides active:")
    for _t, _m in MANUAL_SPLIT_OVERRIDES.items():
        for _d, _f in _m.items():
            print(f"  {_t} {_d} factor={_f}")

# ---- Manual bad-print repairs (targeted) ----
# Modes:
# - "interpolate": one-day midpoint interpolation from neighbors
# - {"mode": "interpolate_window", "dates": [...]}: replace listed dates by
#   linear interpolation anchored by nearest clean observations outside the window.
# - {"mode": "beta_from_underlying", "underlying": "SMR", "beta": 2.058}:
#   reprice ETF on that date from underlying return and beta.
# Keep empty by default; leave hook in place for non-split bad ticks if ever needed.
MANUAL_TR_PRICE_FIXES = {}


def _apply_manual_price_fixes(prices_map, fixes):
    applied = []

    for ticker, dmap in fixes.items():
        s = prices_map.get(ticker)
        if s is None or s.empty:
            continue
        s = s.sort_index().copy()

        # First handle explicit windows.
        for _, spec in dmap.items():
            if not isinstance(spec, dict) or spec.get("mode") != "interpolate_window":
                continue

            dates = [pd.Timestamp(x) for x in spec.get("dates", [])]
            dates = sorted([d for d in dates if d in s.index])
            if not dates:
                continue

            left_idx = s.index[s.index < dates[0]]
            right_idx = s.index[s.index > dates[-1]]
            if left_idx.empty or right_idx.empty:
                continue

            left_t = left_idx[-1]
            right_t = right_idx[0]
            y0 = float(s.loc[left_t])
            y1 = float(s.loc[right_t])
            x0 = float(left_t.value)
            x1 = float(right_t.value)
            if x1 <= x0:
                continue

            for ts in dates:
                old = float(s.loc[ts])
                x = float(ts.value)
                w = (x - x0) / (x1 - x0)
                new = y0 + w * (y1 - y0)
                s.loc[ts] = new
                applied.append((ticker, ts, old, float(new), "interpolate_window"))

        # Then handle single-date fixes.
        for ds, mode in dmap.items():
            ts = pd.Timestamp(ds)
            if ts not in s.index:
                continue

            # Dict mode for date-specific advanced fixes.
            if isinstance(mode, dict):
                m = mode.get("mode")
                if m == "beta_from_underlying":
                    und = str(mode.get("underlying", "")).upper()
                    beta = float(mode.get("beta", 1.0))
                    su = prices_map.get(und)
                    prev_idx = s.index[s.index < ts]
                    if su is None or su.empty or prev_idx.empty:
                        continue

                    prev_t = prev_idx[-1]
                    p_u_t = float(su.asof(ts))
                    p_u_prev = float(su.asof(prev_t))
                    p_e_prev = float(s.loc[prev_t])
                    if p_u_prev <= 0 or p_e_prev <= 0:
                        continue

                    r_u = p_u_t / p_u_prev - 1.0
                    new = max(1e-6, p_e_prev * (1.0 + beta * r_u))
                    old = float(s.loc[ts])
                    s.loc[ts] = new
                    applied.append((ticker, ts, old, float(new), f"beta_from_{und}"))
                continue

            old = float(s.loc[ts])
            prev = s[s.index < ts]
            nxt = s[s.index > ts]

            if mode == "interpolate" and not prev.empty and not nxt.empty:
                new = float((prev.iloc[-1] + nxt.iloc[0]) / 2.0)
            elif not prev.empty:
                new = float(prev.iloc[-1])
            elif not nxt.empty:
                new = float(nxt.iloc[0])
            else:
                continue

            s.loc[ts] = new
            applied.append((ticker, ts, old, new, mode))

        prices_map[ticker] = s

    if applied:
        print("Applied manual price fixes:")
        for ticker, ts, old, new, mode in applied:
            print(f"  {ticker} {ts.date()} [{mode}] {old:,.4f} -> {new:,.4f}")
    else:
        print("No manual price fixes applied.")


_apply_manual_price_fixes(PRICES, MANUAL_TR_PRICE_FIXES)

# Quick sanity check: max absolute Jan-2026 daily return.
for _sym in ["SMR", "SMUP"]:
    _s = PRICES.get(_sym)
    if _s is None or _s.empty:
        continue
    _jan = _s[(_s.index >= pd.Timestamp("2026-01-01")) & (_s.index < pd.Timestamp("2026-02-01"))]
    _r = _jan.pct_change().dropna()
    if not _r.empty:
        _d = _r.abs().idxmax()
        print(f"{_sym} largest Jan return: {_d.date()} | {_r.loc[_d]:.2%}")

# ---- OBFR benchmark (fallback to Fed Funds) ----
try:
    obfr_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=OBFR"
    obfr_df = pd.read_csv(obfr_url)
    obfr_df.columns = [str(c).strip() for c in obfr_df.columns]
    date_col = "DATE" if "DATE" in obfr_df.columns else ("observation_date" if "observation_date" in obfr_df.columns else None)
    if date_col is None or "OBFR" not in obfr_df.columns:
        raise KeyError(f"Unexpected OBFR columns: {list(obfr_df.columns)}")

    obfr_df["date"] = pd.to_datetime(obfr_df[date_col], errors="coerce")
    obfr_df["rate"] = pd.to_numeric(obfr_df["OBFR"], errors="coerce")
    obfr_df = obfr_df.dropna(subset=["date", "rate"]).set_index("date").sort_index()
    FED_FUNDS_DAILY = (obfr_df["rate"] / 100).astype(float)
    print(f"OBFR: {len(FED_FUNDS_DAILY)} daily obs ({date_col})")
except Exception as e_obfr:
    print(f"OBFR fetch failed ({e_obfr}), trying DFF fallback")
    try:
        dff_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFF"
        ff_df = pd.read_csv(dff_url)
        ff_df.columns = [str(c).strip() for c in ff_df.columns]
        date_col = "DATE" if "DATE" in ff_df.columns else ("observation_date" if "observation_date" in ff_df.columns else None)
        if date_col is None or "DFF" not in ff_df.columns:
            raise KeyError(f"Unexpected DFF columns: {list(ff_df.columns)}")

        ff_df["date"] = pd.to_datetime(ff_df[date_col], errors="coerce")
        ff_df["rate"] = pd.to_numeric(ff_df["DFF"], errors="coerce")
        ff_df = ff_df.dropna(subset=["date", "rate"]).set_index("date").sort_index()
        FED_FUNDS_DAILY = (ff_df["rate"] / 100).astype(float)
        print(f"DFF fallback: {len(FED_FUNDS_DAILY)} daily obs ({date_col})")
    except Exception as e_dff:
        print(f"FRED failed ({e_dff}), using static fallback")
        fomc = [(pd.Timestamp("2022-12-14"), 0.0433), (pd.Timestamp("2023-02-01"), 0.0458),
                (pd.Timestamp("2023-03-22"), 0.0483), (pd.Timestamp("2023-05-03"), 0.0508),
                (pd.Timestamp("2023-07-26"), 0.0533), (pd.Timestamp("2024-09-18"), 0.0483),
                (pd.Timestamp("2024-11-07"), 0.0458), (pd.Timestamp("2024-12-18"), 0.0433),
                (pd.Timestamp("2025-01-29"), 0.0433)]
        FED_FUNDS_DAILY = pd.Series({d: r for d, r in fomc}).sort_index()


  20/468 [13s]
  40/468 [21s]
  60/468 [24s]
  80/468 [33s]
  100/468 [38s]
  120/468 [42s]
  140/468 [229s]


$ERX: possibly delisted; no price data found  (1d 1927-05-20 -> 2026-04-25)
$EWY: possibly delisted; no price data found  (1d 1927-05-20 -> 2026-04-25)


  160/468 [233s]
  180/468 [238s]
  200/468 [243s]


$KLAC: possibly delisted; no price data found  (1d 1927-05-20 -> 2026-04-25)
$KTOS: possibly delisted; no price data found  (1d 1927-05-20 -> 2026-04-25)
$LABU: possibly delisted; no price data found  (1d 1927-05-20 -> 2026-04-25)


  220/468 [257s]
  240/468 [261s]
  260/468 [267s]
  280/468 [271s]
  300/468 [276s]
  320/468 [278s]
  340/468 [281s]
  360/468 [283s]
  380/468 [287s]
  400/468 [292s]
  420/468 [294s]
  440/468 [297s]
  460/468 [300s]
Got 459/468 [304.2s]
Manual split overrides active:
  SMUP 2026-01-26 factor=10.0
  EOSU 2026-04-08 factor=25.0
No manual price fixes applied.
SMR largest Jan return: 2026-01-05 | 15.14%
OBFR: 2549 daily obs (observation_date)


In [251]:
# ---- Inception dates + full universe (post–start_date listings join when data exists) ----
if "CFG" not in globals():
    if "cfg" in globals() and isinstance(cfg, dict):
        CFG = cfg
    else:
        raise RuntimeError("CFG is not defined in this kernel. Run the 'Configuration' cell first, then rerun this cell.")

start_ts = pd.Timestamp(CFG["start_date"])

INCEPTION = {}
for etf, und, bv in CANDIDATES:
    s_e = PRICES.get(etf)
    s_u = PRICES.get(und)
    if s_e is not None and s_u is not None:
        first = max(s_e.index[0], s_u.index[0])
        INCEPTION[(etf, und)] = first

UNIVERSE = [(e, u, b) for e, u, b in CANDIDATES if (e, u) in INCEPTION]
UNIVERSE.sort(key=lambda x: INCEPTION[(x[0], x[1])])

RAMP_WEEKS = 4  # weekly rebalances to scale new listings from 1/RAMP_WEEKS → full weight

n_at_start = sum(1 for e, u, b in UNIVERSE if INCEPTION[(e, u)] <= start_ts)
print(f"Pairs with overlapping price data: {len(INCEPTION)} / {len(CANDIDATES)}")
print(f"Full UNIVERSE: {len(UNIVERSE)} pairs (v7: incremental engine)")
print(f"Tradeable at {CFG['start_date']}: {n_at_start} pairs | RAMP_WEEKS={RAMP_WEEKS}")
print()
for i, (e, u, b) in enumerate(UNIVERSE, 1):
    inc = INCEPTION[(e, u)].strftime("%Y-%m-%d")
    tag = "  [post-start]" if INCEPTION[(e, u)] > start_ts else ""
    print(f"  {i:3d}  {e:5s}  {u:5s}  {b:.3f}  {inc}{tag}")


Pairs with overlapping price data: 270 / 279
Full UNIVERSE: 270 pairs (v7: incremental engine)
Tradeable at 2023-01-01: 24 pairs | RAMP_WEEKS=4

    1  YINN   FXI    2.975  2009-12-03
    2  SOXL   SOXX   2.964  2010-03-11
    3  SOXS   SOXX   2.991  2010-03-11
    4  NUGT   GDX    1.976  2010-12-08
    5  JNUG   GDXJ   1.974  2013-10-03
    6  CHAU   ASHR   1.966  2015-04-16
    7  GUSH   XOP    2.010  2015-05-29
    8  CWEB   KWEB   1.988  2016-11-10
    9  MEXX   EWW    2.956  2017-05-03
   10  HIBL   SPHB   3.007  2019-11-07
   11  GDXU   GDX    3.041  2020-12-03
   12  UVIX   SVIX   2.047  2022-03-30
   13  TARK   ARKK   1.979  2022-05-02
   14  TSLQ   TSLA   1.897  2022-07-14
   15  TSLL   TSLA   1.995  2022-08-09
   16  AAPU   AAPL   1.986  2022-08-09
   17  CONL   COIN   1.988  2022-08-10
   18  AAPB   AAPL   1.990  2022-08-10
   19  GGLL   GOOGL  1.996  2022-09-07
   20  AMZU   AMZN   1.989  2022-09-07
   21  MSFU   MSFT   1.992  2022-09-07
   22  BABX   BABA   2.007  2022-12-

## Position weights (`PAIR_WEIGHTS`)

- **CSV path:** `notebooks/data/backtest/Diamond_Creek_Backtest_position_weights.csv`
- **Column:** `weight` — applied when `(etf, und)` matches a row in `UNIVERSE` (symbols via `norm_sym`).
- **Keys:** the v15 engine resolves weights with `PAIR_WEIGHTS.get(etf, 1/n_pairs)` inside `_frac_rows_for_day`; we set **every ETF in `UNIVERSE` explicitly** (default **0** if missing from the CSV) so there is **no silent equal-weight fallback**.
- **Normalization:** weights are **not** forced to sum to 1. In `_apply_allocs`, `wcomb = wb * rp` and `tw = sum(wcomb)` rescales allocations each rebalance — **only relative weights** matter.


In [252]:
WEIGHTS_CSV = Path("data/backtest/Diamond_Creek_Backtest_position_weights.csv")
_wdf = pd.read_csv(WEIGHTS_CSV)
_wdf["etf"] = _wdf["etf"].astype(str).map(norm_sym)
_wdf["und"] = _wdf["und"].astype(str).map(norm_sym)
_wdf["weight"] = pd.to_numeric(_wdf["weight"], errors="coerce").fillna(0.0)
_univ_keys = {(norm_sym(e), norm_sym(u)) for e, u, _ in UNIVERSE}
PAIR_WEIGHTS = {norm_sym(e): 0.0 for e, _, _ in UNIVERSE}
for _, r in _wdf.iterrows():
    key = (r["etf"], r["und"])
    if key not in _univ_keys:
        continue
    PAIR_WEIGHTS[r["etf"]] = float(r["weight"])
_pos = sum(1 for v in PAIR_WEIGHTS.values() if v > 0)
_raw = float(sum(PAIR_WEIGHTS.values()))
print(f"PAIR_WEIGHTS from CSV: {_pos} positive ETFs, raw sum {_raw:.6f} (relative-only; engine normalizes wcomb).")


PAIR_WEIGHTS from CSV: 242 positive ETFs, raw sum 0.979454 (relative-only; engine normalizes wcomb).


## Cost Functions


In [253]:
def trade_cost(tusd, px, cfg):
    if abs(tusd) < 1 or px <= 0:
        return 0.0
    sh = abs(tusd) / px
    comm = max(cfg["clear_street_comm_min"],
               min(sh * cfg["clear_street_comm_per_share"], abs(tusd) * cfg["clear_street_comm_max_pct"]))
    return comm + abs(tusd) * cfg["slippage_bps"] / 10000

def margin_debit_interest(debit, benchmark_rate, spreads):
    if debit <= 0:
        return 0.0
    daycount = float(CFG.get("financing_daycount", TRADING_DAYS))
    remaining, interest, prev = debit, 0.0, 0.0
    for threshold, spread in spreads:
        rate = max(0.0, benchmark_rate + spread)
        amt = min(remaining, threshold - prev)
        if amt > 0:
            interest += amt * rate / daycount
            remaining -= amt
        prev = threshold
        if remaining <= 0:
            break
    return interest

def short_credit_interest(short_notional, benchmark_rate, credit_spread):
    if short_notional <= 0:
        return 0.0
    if not bool(CFG.get("enable_short_credit_income", True)):
        return 0.0
    daycount = float(CFG.get("financing_daycount", TRADING_DAYS))
    rate = max(0.0, benchmark_rate + credit_spread)
    return short_notional * rate / daycount

def _get_fed_funds(dt):
    val = FED_FUNDS_DAILY.asof(dt)
    return 0.04 if pd.isna(val) else float(val)

print("Cost functions OK")


Cost functions OK


## Backtest Engine — v8 base engine (incremental adds, dynamic target gross)

- **Day 0:** Incumbents only, gross target **`NAV × V7_TARGET_LEV_START`** (same as a 4.5× launch book).
- **Later rebalances:** Apply **v5-style gross control**: if gross drift exceeds `gross_dead_band_pct`, run full `_apply_allocs` for all eligible pairs; otherwise beta-hedge shorts inside the dead band.
- **Post-start adds:** `_apply_one_pair` toward `tgt_gross_nav × (wcomb/Σw)` for active post-start names, with `RAMP_WEEKS` ramp-in.
- **Target gross:** `tgt_gross_nav = NAV × target_lev` with `target_lev` linear in **fraction of post-start pairs** with gross > \$1.


In [254]:
spy = PRICES.get("SPY")
tdays = spy.index[spy.index >= start_ts].sort_values()

s = pd.Series(range(len(tdays)), index=tdays)
rebal_idx = s.resample("W-FRI").last().dropna()
rebal_days = set(tdays[int(i)] for i in rebal_idx.values)

n_pairs = len(UNIVERSE)
INC_ETFS = frozenset(
    etf for etf, und, bv in UNIVERSE if INCEPTION[(etf, und)] <= start_ts
)
POST_ETFS = frozenset(
    etf for etf, und, bv in UNIVERSE if INCEPTION[(etf, und)] > start_ts
)
B3_ETFS = frozenset(str(x).upper().strip().replace(".", "-") for x in (globals().get("BUCKET3_ETFS", set()) or set()))
B3_KEYS = frozenset((etf, und) for etf, und, _ in UNIVERSE if str(etf).upper().strip().replace(".", "-") in B3_ETFS)
B3_INC_KEYS = frozenset(k for k in B3_KEYS if INCEPTION[k] <= start_ts)

print(f"Trading days: {len(tdays)} | Rebalance days: {len(rebal_days)} (weekly)")
print(f"Pairs: {n_pairs} | Incumbents (≤ start): {len(INC_ETFS)} | Post-start: {len(POST_ETFS)}")
print(f"Bucket3 configured: {len(B3_ETFS)} ETFs | in-universe: {len(B3_KEYS)} | active at start: {len(B3_INC_KEYS)}")
print(f"Start: {tdays[0].strftime('%Y-%m-%d')} | End: {tdays[-1].strftime('%Y-%m-%d')}")
print(
    f"v8 target gross multiple: {V7_TARGET_LEV_START}x → {V7_TARGET_LEV_END}x "
    f"as post-start pairs join (v5-style incumbent hedging + dead-band gross control)"
)

if "PAIR_WEIGHTS" not in dir():
    PAIR_WEIGHTS = {e: 1.0 / n_pairs for e, _, _ in UNIVERSE}
    print(f"Using equal weights: {1.0/n_pairs:.4%} per pair")
else:
    print(f"Using custom PAIR_WEIGHTS ({len(PAIR_WEIGHTS)} pairs, top weight: {max(PAIR_WEIGHTS.values()):.2%})")

# Reconciliation controls for matching external sheets exactly.
# Provide overrides in a prior cell, e.g.:
# EXACT_SHARE_OVERRIDES = {
#   ("NVDL", "NVDA"): {"long_sh": 12345, "short_sh": -67890},
#   ("2026-01-02", "NVDL", "NVDA"): {"long_sh": 12345, "short_sh": -67890},
# }
if "EXACT_SHARE_OVERRIDES" not in dir():
    EXACT_SHARE_OVERRIDES = {}

# Optional rounding mode for share conversion: "round" (default), "floor", or "ceil".
if "SHARE_ROUNDING_MODE" not in dir():
    SHARE_ROUNDING_MODE = "round"


def _shares_from_usd(usd: float, px: float, side: str) -> int:
    if px <= 0 or usd <= 0:
        return 0
    raw = usd / px
    mode = str(SHARE_ROUNDING_MODE).lower()
    if mode == "floor":
        sh = int(np.floor(raw))
    elif mode == "ceil":
        sh = int(np.ceil(raw))
    else:
        sh = int(round(raw))
    return sh if side == "long" else -sh


def _apply_exact_share_override(today, etf: str, und: str, lsh: int, ssh: int) -> tuple[int, int]:
    day_key = str(pd.Timestamp(today).date()) if today is not None else None
    ov = None
    if day_key is not None:
        ov = EXACT_SHARE_OVERRIDES.get((day_key, etf, und))
    if ov is None:
        ov = EXACT_SHARE_OVERRIDES.get((etf, und))
    if ov is None:
        ov = EXACT_SHARE_OVERRIDES.get(etf)
    if isinstance(ov, dict):
        lsh = int(ov.get("long_sh", lsh))
        ssh = int(ov.get("short_sh", ssh))
    return lsh, ssh

ALL_BT = {}
ALL_PAIR_PNL = {}
ALL_PAIR_BORROW = {}
ALL_PAIR_NET = {}
ALL_PAIR_GROSS = {}
ALL_PAIR_DAILY = {}

for gross_lev in LEVERAGE_RUNS:
    print(f"\n{'='*60}")
    print(
        f"  v7 RUN (store key {gross_lev}x) | dynamic tgt gross {V7_TARGET_LEV_START}x–{V7_TARGET_LEV_END}x"
    )
    print(f"{'='*60}")

    cash = float(CFG["capital_usd"])
    holdings = {}
    pair_pos = {}
    pair_pnl = {}
    pair_borrow = {}
    daily_rows = []
    pair_daily_rows = []
    tot_costs = tot_borrow = tot_margin_debit = tot_margin_credit = 0.0
    rebal_count = 0
    pair_entry_rebal: dict[tuple[str, str], int] = {}
    prev_px = {}
    # Excel-style tradeflow state by pair leg:
    # level = cumulative trade cashflow + current marked value
    pair_long_trade_cum = {}
    pair_short_trade_cum = {}
    pair_prev_long_level = {}
    pair_prev_short_level = {}
    cum_long_pnl = cum_short_pnl = 0.0
    pair_net_rows = []
    pair_gross_rows = []

    def _execute_trade(sym, tgt_sh, px_map):
        global cash
        cur_sh = holdings.get(sym, 0)
        delta = tgt_sh - cur_sh
        p = px_map.get(sym, 0)
        trade_usd = abs(delta * p)
        if p <= 0 or trade_usd < 1:
            return 0.0, 0.0
        cash -= delta * p
        c = trade_cost(trade_usd, p, CFG)
        cash -= c
        if abs(tgt_sh) < 1:
            holdings.pop(sym, None)
        else:
            holdings[sym] = tgt_sh
        return trade_usd, c

    def _frac_rows_for_day(today, R: int):
        rows = []
        for etf, und, bv in UNIVERSE:
            key = (etf, und)
            if INCEPTION[key] > today:
                continue
            if INCEPTION[key] <= start_ts:
                rp = 1.0
            else:
                if key not in pair_entry_rebal:
                    pair_entry_rebal[key] = R
                rp = min(1.0, (R - pair_entry_rebal[key] + 1) / float(RAMP_WEEKS))
            wb = PAIR_WEIGHTS.get(etf, 1.0 / n_pairs)
            rows.append((etf, und, bv, wb * rp))
        return rows

    def _pair_gross_usd(ek, px_map):
        pos = pair_pos.get(ek)
        if not pos:
            return 0.0
        uk = pos["und"]
        return abs(float(pos.get("long_sh", 0.0))) * px_map.get(
            uk, 0
        ) + abs(float(pos.get("short_sh", 0.0))) * px_map.get(ek, 0)

    def _target_lev_for_book(px_map):
        if not POST_ETFS:
            return float(V7_TARGET_LEV_START)
        on = sum(1 for ek in POST_ETFS if _pair_gross_usd(ek, px_map) > 1.0)
        prog = on / float(len(POST_ETFS))
        return V7_TARGET_LEV_START + (V7_TARGET_LEV_END - V7_TARGET_LEV_START) * prog

    def _apply_allocs(tgt_gross: float, frac_rows, px_map: dict, today=None) -> dict:
        tw = sum(r[3] for r in frac_rows)
        tgt_shares: dict = {}
        if tw <= 0:
            return tgt_shares
        for etf, und, bv, wcomb in frac_rows:
            frac = wcomb / tw
            alloc = tgt_gross * frac
            p_u, p_e = px_map.get(und, 0), px_map.get(etf, 0)
            if p_u <= 0 or p_e <= 0:
                continue
            babs = max(abs(bv), 1e-9)
            hr = 1.0 / babs
            is_bucket3 = str(etf).upper().strip().replace(".", "-") in B3_ETFS
            und_leg_usd = alloc / (1.0 + hr)
            etf_leg_usd = alloc - und_leg_usd
            if is_bucket3:
                lsh = _shares_from_usd(und_leg_usd, p_u, side="short")
                ssh = _shares_from_usd(etf_leg_usd, p_e, side="short")
            else:
                lsh = _shares_from_usd(und_leg_usd, p_u, side="long")
                ssh = _shares_from_usd(etf_leg_usd, p_e, side="short")
            lsh, ssh = _apply_exact_share_override(today, etf, und, lsh, ssh)
            pair_pos[etf] = {"und": und, "long_sh": lsh, "short_sh": ssh}
            if etf not in pair_pnl:
                pair_pnl[etf] = {"und": und, "long": 0.0, "short": 0.0}
            if etf not in pair_borrow:
                pair_borrow[etf] = 0.0
            tgt_shares[und] = tgt_shares.get(und, 0) + lsh
            tgt_shares[etf] = tgt_shares.get(etf, 0) + ssh
        return tgt_shares

    def _apply_one_pair(
        tgt_shares: dict, etf, und, bv, alloc_usd: float, px_map: dict, today=None
    ) -> None:
        p_u, p_e = px_map.get(und, 0), px_map.get(etf, 0)
        if p_u <= 0 or p_e <= 0 or alloc_usd <= 0:
            return
        babs = max(abs(bv), 1e-9)
        hr = 1.0 / babs
        und_leg_usd = alloc_usd / (1.0 + hr)
        etf_leg_usd = alloc_usd - und_leg_usd
        is_bucket3 = str(etf).upper().strip().replace(".", "-") in B3_ETFS
        if is_bucket3:
            lsh = _shares_from_usd(und_leg_usd, p_u, side="short")
            ssh = _shares_from_usd(etf_leg_usd, p_e, side="short")
        else:
            lsh = _shares_from_usd(und_leg_usd, p_u, side="long")
            ssh = _shares_from_usd(etf_leg_usd, p_e, side="short")
        lsh, ssh = _apply_exact_share_override(today, etf, und, lsh, ssh)
        pair_pos[etf] = {"und": und, "long_sh": lsh, "short_sh": ssh}
        if etf not in pair_pnl:
            pair_pnl[etf] = {"und": und, "long": 0.0, "short": 0.0}
        if etf not in pair_borrow:
            pair_borrow[etf] = 0.0
        tgt_shares[und] = tgt_shares.get(und, 0) + lsh
        tgt_shares[etf] = tgt_shares.get(etf, 0) + ssh

    for di, today in enumerate(tdays):
        px = {}
        for sym, ser in PRICES.items():
            if today in ser.index:
                px[sym] = float(ser.loc[today])
            else:
                prior = ser[ser.index <= today]
                if not prior.empty:
                    px[sym] = float(prior.iloc[-1])

        # Execution timing (for Excel tie-out use same-close execution).
        # - same_close: fills at today's close (default)
        # - prior_close: fills at previous close when available
        exec_timing = str(CFG.get("rebalance_execution_timing", "same_close")).strip().lower()
        if exec_timing in {"same_close", "close", "today_close"}:
            px_trade = dict(px)
        elif exec_timing in {"prior_close", "previous_close", "t_minus_1_close"}:
            px_trade = dict(prev_px) if (di > 0 and prev_px) else dict(px)
        else:
            raise ValueError(
                f"Unsupported rebalance_execution_timing={exec_timing!r}; "
                "use same_close or prior_close"
            )

        pair_day = {}
        for ek, pos in pair_pos.items():
            uk = pos["und"]
            lsh = float(pos.get("long_sh", 0))
            ssh = float(pos.get("short_sh", 0))
            p_u = float(px.get(uk, 0.0) or 0.0)
            p_e = float(px.get(ek, 0.0) or 0.0)
            long_n_sig = lsh * p_u
            short_n_sig = ssh * p_e
            und_long_val = max(lsh, 0.0) * p_u
            und_short_val = max(-lsh, 0.0) * p_u
            etf_short_val = max(-ssh, 0.0) * p_e
            short_fin_basis = und_short_val + etf_short_val
            pair_day[ek] = {
                "date": today,
                "etf": ek,
                "under": uk,
                "long_sh": lsh,
                "short_sh": ssh,
                "underlying_price": p_u if p_u > 0 else float("nan"),
                "etf_price": p_e if p_e > 0 else float("nan"),
                "long_notional_usd": long_n_sig,
                "short_notional_usd": short_n_sig,
                "short_financing_basis_usd": short_fin_basis,
                "long_margin_basis_usd": und_long_val,
                "gross_notional_usd": abs(lsh) * p_u + abs(ssh) * p_e,
                "net_notional_usd": long_n_sig + short_n_sig,
                "borrow_rate_annual": float(BORROW_MAP.get(ek, CFG["fallback_borrow_rate"])),
                "daily_long_pnl_usd": 0.0,
                "daily_short_pnl_usd": 0.0,
                "daily_borrow_cost_usd": 0.0,
                "daily_underlying_borrow_cost_usd": 0.0,
                "daily_margin_debit_cost_usd": 0.0,
                "daily_short_credit_income_usd": 0.0,
                "daily_net_financing_cost_usd": 0.0,
                "daily_txn_cost_usd": 0.0,
                "daily_turnover_usd": 0.0,
            }

        daily_long_pnl = daily_short_pnl = 0.0
        for ek, pos in pair_pos.items():
            uk = pos["und"]
            lsh = float(pos.get("long_sh", 0.0))
            ssh = float(pos.get("short_sh", 0.0))

            # Excel-style level method by leg:
            # level = cumulative trade cashflow + marked current value.
            long_level = float(pair_long_trade_cum.get(ek, 0.0) + lsh * px.get(uk, 0))
            short_level = float(pair_short_trade_cum.get(ek, 0.0) + ssh * px.get(ek, 0))

            prev_long_level = pair_prev_long_level.get(ek)
            prev_short_level = pair_prev_short_level.get(ek)

            if di == 0 or prev_long_level is None or prev_short_level is None:
                l_pnl = 0.0
                s_pnl = 0.0
            else:
                l_pnl = long_level - float(prev_long_level)
                s_pnl = short_level - float(prev_short_level)

            # Keep legacy MTM move for audit/debug comparison.
            if di > 0 and prev_px:
                dp_u = px.get(uk, 0) - prev_px.get(uk, 0)
                dp_e = px.get(ek, 0) - prev_px.get(ek, 0)
                l_pnl_mtm = lsh * dp_u if lsh > 0 else 0.0
                s_pnl_mtm = ssh * dp_e if ssh < 0 else 0.0
            else:
                l_pnl_mtm = 0.0
                s_pnl_mtm = 0.0

            pair_prev_long_level[ek] = long_level
            pair_prev_short_level[ek] = short_level

            daily_long_pnl += l_pnl
            daily_short_pnl += s_pnl
            pair_pnl[ek]["long"] += l_pnl
            pair_pnl[ek]["short"] += s_pnl

            if ek in pair_day:
                pair_day[ek]["daily_long_pnl_usd"] = float(l_pnl)
                pair_day[ek]["daily_short_pnl_usd"] = float(s_pnl)
                pair_day[ek]["daily_long_pnl_usd_mtm"] = float(l_pnl_mtm)
                pair_day[ek]["daily_short_pnl_usd_mtm"] = float(s_pnl_mtm)
                pair_day[ek]["excel_long_pnl_level_usd"] = float(long_level)
                pair_day[ek]["excel_short_pnl_level_usd"] = float(short_level)

        cum_long_pnl += daily_long_pnl
        cum_short_pnl += daily_short_pnl

        mtm = sum(sh * px.get(sym, 0) for sym, sh in holdings.items())
        nav = cash + mtm

        borrow_d = 0.0
        for ek, pos in pair_pos.items():
            sh_short_etf = float(pos.get("short_sh", 0))
            sh_short_und = float(pos.get("long_sh", 0))
            b_etf = 0.0
            b_und = 0.0
            if sh_short_etf < 0:
                b_etf = (
                    abs(sh_short_etf)
                    * px.get(ek, 0)
                    * BORROW_MAP.get(ek, CFG["fallback_borrow_rate"])
                    / TRADING_DAYS
                )
            if sh_short_und < 0:
                und_sym = pos.get("und")
                b_und = (
                    abs(sh_short_und)
                    * px.get(und_sym, 0)
                    * BORROW_MAP.get(und_sym, CFG["fallback_borrow_rate"])
                    / TRADING_DAYS
                )
            b = float(b_etf + b_und)
            if b > 0:
                borrow_d += b
                pair_borrow[ek] = pair_borrow.get(ek, 0.0) + b
                if ek in pair_day:
                    pair_day[ek]["daily_borrow_cost_usd"] = float(b)
                    pair_day[ek]["daily_underlying_borrow_cost_usd"] = float(b_und)
        cash -= borrow_d
        tot_borrow += borrow_d

        ff = _get_fed_funds(today)
        margin_debit_d = 0.0
        margin_credit_d = 0.0

        l_notional = sum(max(0.0, d["long_notional_usd"]) for d in pair_day.values())
        s_notional = sum(d["short_notional_usd"] for d in pair_day.values())

        # Financing base override:
        # charge debit on full long notional minus NAV (ignore short proceeds offset).
        net_debit_balance = max(0.0, l_notional - nav)
        margin_debit_d = margin_debit_interest(
            net_debit_balance, ff, CFG["margin_debit_spreads"]
        )

        # Allocate book-level debit back to pairs for additive diagnostics.
        for ek, d in pair_day.items():
            debit_alloc = (
                margin_debit_d * (max(0.0, d["long_notional_usd"]) / l_notional)
                if l_notional > 0
                else 0.0
            )
            credit = short_credit_interest(
                float(d["short_financing_basis_usd"]), ff, CFG["credit_spread"]
            )
            d["daily_margin_debit_cost_usd"] = float(debit_alloc)
            d["daily_short_credit_income_usd"] = float(credit)
            d["daily_net_financing_cost_usd"] = float(debit_alloc - credit)
            margin_credit_d += credit

        cash -= margin_debit_d
        tot_margin_debit += margin_debit_d

        cash += margin_credit_d
        tot_margin_credit += margin_credit_d

        mtm = sum(sh * px.get(sym, 0) for sym, sh in holdings.items())
        nav = cash + mtm

        trade_turnover = 0.0
        dc = 0.0
        is_rebal = (today in rebal_days) or (di == 0)
        pair_pos_before_rebal = {
            ek: {
                "und": pos["und"],
                "long_sh": float(pos.get("long_sh", 0)),
                "short_sh": float(pos.get("short_sh", 0)),
            }
            for ek, pos in pair_pos.items()
        }

        if is_rebal:
            rebal_count += 1
            frac_rows = _frac_rows_for_day(today, rebal_count)
            db = CFG["dead_band_pct"]
            tw = sum(r[3] for r in frac_rows)
            gdb = CFG.get("gross_dead_band_pct", 0.05)
            b3_active_today = sorted({e for e, _, _, _ in frac_rows if str(e).upper().strip().replace(".", "-") in B3_ETFS})
            print(f"[bucket3] universe={len(B3_KEYS)} | active_on_{today.strftime('%Y-%m-%d')}={len(b3_active_today)}")

            target_lev_t = _target_lev_for_book(px)
            tgt_gross_nav = nav * target_lev_t

            actual_gross = sum(
                abs(float(p.get("long_sh", 0))) * px.get(p["und"], 0)
                + abs(float(p.get("short_sh", 0))) * px.get(ek, 0)
                for ek, p in pair_pos.items()
            )
            gross_drift = (
                abs(actual_gross - tgt_gross_nav) / tgt_gross_nav
                if tgt_gross_nav > 0
                else 0
            )

            tgt_shares: dict = {}

            if di == 0:
                rows0 = [r for r in frac_rows if r[0] in INC_ETFS]
                tw0 = sum(r[3] for r in rows0)
                if tw0 <= 0:
                    raise RuntimeError("v7 day 0: no incumbent frac_rows")
                tgt0 = nav * float(V7_TARGET_LEV_START)
                tgt_shares = _apply_allocs(tgt0, rows0, px_trade, today=today)
            else:
                if gross_drift > gdb:
                    tgt_shares = _apply_allocs(tgt_gross_nav, frac_rows, px_trade, today=today)
                else:
                    for etf, und, bv in UNIVERSE:
                        if etf not in INC_ETFS:
                            continue
                        pos = pair_pos.get(etf)
                        if pos is None:
                            continue
                        p_u, p_e = px.get(und, 0), px.get(etf, 0)
                        if p_u <= 0 or p_e <= 0:
                            continue
                        babs = max(abs(bv), 1e-9)
                        is_bucket3 = str(etf).upper().strip().replace(".", "-") in B3_ETFS

                        if not is_bucket3:
                            l_val = max(pos["long_sh"], 0) * p_u
                            s_val = max(-pos["short_sh"], 0) * p_e
                            pair_gross = l_val + s_val
                            beta_net = l_val - babs * s_val

                            if pair_gross > 0 and abs(beta_net) / pair_gross > db:
                                target_short_usd = l_val / babs
                                new_ssh = _shares_from_usd(target_short_usd, p_e, side="short")
                                _, new_ssh = _apply_exact_share_override(
                                    today, etf, und, pos["long_sh"], new_ssh
                                )
                                pair_pos[etf]["short_sh"] = new_ssh

                        tgt_shares[und] = tgt_shares.get(und, 0) + pair_pos[etf].get("long_sh", 0)
                        tgt_shares[etf] = tgt_shares.get(etf, 0) + pair_pos[etf].get(
                            "short_sh", 0
                        )

                    if tw > 0 and tgt_gross_nav > 0:
                        for etf, und, bv, wcomb in frac_rows:
                            if etf in INC_ETFS:
                                continue
                            alloc = tgt_gross_nav * (wcomb / tw)
                            _apply_one_pair(tgt_shares, etf, und, bv, alloc, px_trade, today=today)

            for sym in set(tgt_shares) | set(holdings):
                tusd, tc = _execute_trade(sym, tgt_shares.get(sym, 0), px_trade)
                trade_turnover += tusd
                dc += tc

            # Pair-led transaction ledger: compute turnover per pair from share deltas.
            # This intentionally ignores cross-pair netting at a symbol level.
            pair_keys = set(pair_pos_before_rebal) | set(pair_pos)
            for ek in pair_keys:
                new_pos = pair_pos.get(ek)
                old_pos = pair_pos_before_rebal.get(ek)
                und = (
                    new_pos["und"]
                    if new_pos is not None
                    else old_pos["und"]
                    if old_pos is not None
                    else None
                )
                if und is None:
                    continue
                old_l = float(old_pos.get("long_sh", 0)) if old_pos is not None else 0.0
                old_s = float(old_pos.get("short_sh", 0)) if old_pos is not None else 0.0
                new_l = float(new_pos.get("long_sh", 0)) if new_pos is not None else 0.0
                new_s = float(new_pos.get("short_sh", 0)) if new_pos is not None else 0.0
                turn_l = abs(new_l - old_l) * px_trade.get(und, 0)
                turn_s = abs(new_s - old_s) * px_trade.get(ek, 0)
                pair_turn = float(turn_l + turn_s)
                pair_tc = trade_cost(pair_turn, max(px_trade.get(und, 0), 1e-9), CFG) if pair_turn > 0 else 0.0

                # Update Excel-style cumulative trade cashflows from executed share deltas.
                # Buys consume cash (negative), sells generate cash (positive).
                long_trade_cash = -float(new_l - old_l) * px_trade.get(und, 0)
                short_trade_cash = -float(new_s - old_s) * px_trade.get(ek, 0)
                pair_long_trade_cum[ek] = float(pair_long_trade_cum.get(ek, 0.0) + long_trade_cash)
                pair_short_trade_cum[ek] = float(pair_short_trade_cum.get(ek, 0.0) + short_trade_cash)

                # Ensure newly-opened pairs on day 0 are present in pair_day so
                # entry turnover/txn costs are captured in the pair ledger.
                if ek not in pair_day and new_pos is not None:
                    lsh = float(new_pos.get("long_sh", 0.0))
                    ssh = float(new_pos.get("short_sh", 0.0))
                    p_u = float(px.get(und, 0.0) or 0.0)
                    p_e = float(px.get(ek, 0.0) or 0.0)
                    long_n_sig = lsh * p_u
                    short_n_sig = ssh * p_e
                    und_long_val = max(lsh, 0.0) * p_u
                    und_short_val = max(-lsh, 0.0) * p_u
                    etf_short_val = max(-ssh, 0.0) * p_e
                    short_fin_basis = und_short_val + etf_short_val
                    pair_day[ek] = {
                        "date": today,
                        "etf": ek,
                        "under": und,
                        "long_sh": lsh,
                        "short_sh": ssh,
                        "underlying_price": p_u if p_u > 0 else float("nan"),
                        "etf_price": p_e if p_e > 0 else float("nan"),
                        "long_notional_usd": long_n_sig,
                        "short_notional_usd": short_n_sig,
                        "short_financing_basis_usd": short_fin_basis,
                        "long_margin_basis_usd": und_long_val,
                        "gross_notional_usd": abs(lsh) * p_u + abs(ssh) * p_e,
                        "net_notional_usd": long_n_sig + short_n_sig,
                        "borrow_rate_annual": float(BORROW_MAP.get(ek, CFG["fallback_borrow_rate"])),
                        "daily_long_pnl_usd": 0.0,
                        "daily_short_pnl_usd": 0.0,
                        "daily_borrow_cost_usd": 0.0,
                        "daily_underlying_borrow_cost_usd": 0.0,
                        "daily_margin_debit_cost_usd": 0.0,
                        "daily_short_credit_income_usd": 0.0,
                        "daily_net_financing_cost_usd": 0.0,
                        "daily_txn_cost_usd": 0.0,
                        "daily_turnover_usd": 0.0,
                    }

                if ek in pair_day:
                    pair_day[ek]["daily_turnover_usd"] = pair_turn
                    pair_day[ek]["daily_txn_cost_usd"] = float(pair_tc)
                    pair_day[ek]["excel_long_trades_usd"] = float(pair_long_trade_cum.get(ek, 0.0))
                    pair_day[ek]["excel_short_trades_usd"] = float(pair_short_trade_cum.get(ek, 0.0))

        tot_costs += dc

        net_row = {"date": today}
        gross_row = {"date": today}
        for ek, pos in pair_pos.items():
            uk = pos["und"]
            und_signed = float(pos.get("long_sh", 0.0)) * px.get(uk, 0)
            etf_signed = float(pos.get("short_sh", 0.0)) * px.get(ek, 0)
            und_abs = abs(float(pos.get("long_sh", 0.0))) * px.get(uk, 0)
            etf_abs = abs(float(pos.get("short_sh", 0.0))) * px.get(ek, 0)
            net_row[f"{uk}/{ek}"] = und_signed + etf_signed
            gross_row[f"{uk}/{ek}"] = und_abs + etf_abs
        pair_net_rows.append(net_row)
        pair_gross_rows.append(gross_row)

        mtm = sum(sh * px.get(sym, 0) for sym, sh in holdings.items())
        nv = cash + mtm
        l = sum(sh * px.get(sym, 0) for sym, sh in holdings.items() if sh > 0)
        s = sum(abs(sh) * px.get(sym, 0) for sym, sh in holdings.items() if sh < 0)
        actual_gross = l + s

        target_lev_t = _target_lev_for_book(px)
        tgt_gross_nav = nv * target_lev_t

        daily_rows.append(
            {
                "date": today,
                "nav": nv,
                "cash": cash,
                "long_notional": l,
                "short_notional": s,
                "gross_notional": actual_gross,
                "net_notional": l - s,
                "actual_leverage": actual_gross / nv if nv > 0 else np.nan,
                "target_lev": target_lev_t,
                "tgt_gross_nav": tgt_gross_nav,
                "trade_turnover": trade_turnover,
                "is_rebal": int(is_rebal),
                "fed_funds_rate": ff,
                "net_debit_balance_usd": net_debit_balance,
                "cum_costs": tot_costs,
                "cum_borrow": tot_borrow,
                "cum_margin_debit": tot_margin_debit,
                "cum_margin_credit": tot_margin_credit,
                "daily_borrow": borrow_d,
                "daily_long_pnl": daily_long_pnl,
                "daily_short_pnl": daily_short_pnl,
                "cum_long_pnl": cum_long_pnl,
                "cum_short_pnl": cum_short_pnl,
            }
        )

        for ek, d in pair_day.items():
            d["fed_funds_rate"] = float(ff)
            d["is_rebal"] = int(is_rebal)
            d["daily_pair_gross_trading_pnl_usd"] = float(
                d["daily_long_pnl_usd"] + d["daily_short_pnl_usd"]
            )
            d["daily_total_cost_usd"] = float(
                d["daily_borrow_cost_usd"]
                + d["daily_net_financing_cost_usd"]
                + d["daily_txn_cost_usd"]
            )
            d["daily_pair_net_pnl_usd"] = float(
                d["daily_pair_gross_trading_pnl_usd"] - d["daily_total_cost_usd"]
            )
            pair_daily_rows.append(d)

        prev_px = dict(px)

        if di % 63 == 0:
            ds = today.strftime("%Y-%m-%d")
            tag = " REBAL" if is_rebal else ""
            print(
                f"  {ds}  NAV=${nv:>12,.0f}  Gross=${actual_gross:>12,.0f}  "
                f"L=${l:>10,.0f}  S=${s:>10,.0f}  Cash=${cash:>10,.0f}  "
                f"FF={ff:.2%}{tag}  tgtL={target_lev_t:.2f}x"
            )

    bt = pd.DataFrame(daily_rows).set_index("date")
    bt.index = pd.to_datetime(bt.index)
    ALL_BT[gross_lev] = bt
    ALL_PAIR_PNL[gross_lev] = dict(pair_pnl)
    ALL_PAIR_BORROW[gross_lev] = dict(pair_borrow)
    pnet_df = pd.DataFrame(pair_net_rows)
    if not pnet_df.empty:
        pnet_df = pnet_df.set_index("date")
    ALL_PAIR_NET[gross_lev] = pnet_df
    pgross_df = pd.DataFrame(pair_gross_rows)
    if not pgross_df.empty:
        pgross_df = pgross_df.set_index("date")
    ALL_PAIR_GROSS[gross_lev] = pgross_df

    pair_daily_df = pd.DataFrame(pair_daily_rows)
    if not pair_daily_df.empty:
        pair_daily_df["date"] = pd.to_datetime(pair_daily_df["date"])
        pair_daily_df = pair_daily_df.sort_values(["date", "etf"]).reset_index(drop=True)
    ALL_PAIR_DAILY[gross_lev] = pair_daily_df

    n_days = len(bt)
    years = n_days / TRADING_DAYS
    nav_ser = bt["nav"]
    cagr = (nav_ser.iloc[-1] / nav_ser.iloc[0]) ** (1 / years) - 1
    vol = nav_ser.pct_change().dropna().std() * (TRADING_DAYS**0.5)
    dd = (nav_ser - nav_ser.cummax()) / nav_ser.cummax()

    print(f"\n  {gross_lev}x | Rebals: {rebal_count} | NAV: ${nav_ser.iloc[0]:,.0f} -> ${nav_ser.iloc[-1]:,.0f}")
    print(f"  CAGR: {cagr:.2%} | Vol: {vol:.2%} | Sharpe: {cagr/vol:.2f} | MaxDD: {dd.min():.1%}")
    print(f"  Costs: ${tot_costs:,.0f} | Borrow: ${tot_borrow:,.0f}")
    print(f"  Margin debit (net debit base): ${tot_margin_debit:,.0f}")
    print(f"  Short credit (on proceeds):    ${tot_margin_credit:,.0f}")
    print(f"  Net financing: ${tot_margin_debit - tot_margin_credit + tot_borrow:,.0f}")
    print(f"  Long P&L: ${cum_long_pnl:,.0f} | Short P&L: ${cum_short_pnl:,.0f}")
    print(f"  Avg lev: {bt['actual_leverage'].mean():.2f}x | Avg cash: ${bt['cash'].mean():,.0f}")
    print(
        f"  Target lev (rebal days): min={bt['target_lev'].min():.2f}x max={bt['target_lev'].max():.2f}x"
    )

print(f"\n{'='*60}")
print("  v7 ALL RUNS COMPLETE")
print(f"{'='*60}")


Trading days: 830 | Rebalance days: 173 (weekly)
Pairs: 270 | Incumbents (≤ start): 24 | Post-start: 246
Bucket3 configured: 10 ETFs | in-universe: 10 | active at start: 2
Start: 2023-01-03 | End: 2026-04-24
v8 target gross multiple: 4.5x → 5x as post-start pairs join (v5-style incumbent hedging + dead-band gross control)
Using custom PAIR_WEIGHTS (270 pairs, top weight: 5.00%)

  v7 RUN (store key 4.75x) | dynamic tgt gross 4.5x–5x
[bucket3] universe=10 | active_on_2023-01-03=2
  2023-01-03  NAV=$   9,947,431  Gross=$  41,976,207  L=$27,503,391  S=$14,472,816  Cash=$-3,083,144  FF=4.32% REBAL  tgtL=4.50x
[bucket3] universe=10 | active_on_2023-01-06=2
[bucket3] universe=10 | active_on_2023-01-13=2
[bucket3] universe=10 | active_on_2023-01-20=2
[bucket3] universe=10 | active_on_2023-01-27=2
[bucket3] universe=10 | active_on_2023-02-03=2
[bucket3] universe=10 | active_on_2023-02-10=2
[bucket3] universe=10 | active_on_2023-02-17=2
[bucket3] universe=10 | active_on_2023-02-24=2
[bucket3] u

## Performance summary (fixed `PAIR_WEIGHTS` from CSV)


In [255]:
def perf(nav):
    r = nav.pct_change().dropna()
    ny = len(nav) / TRADING_DAYS
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1 / ny) - 1
    vol = r.std() * (TRADING_DAYS**0.5)
    sr = cagr / vol if vol > 0 else 0
    neg = r[r < 0]
    dv = neg.std() * (TRADING_DAYS**0.5)
    sortino = cagr / dv if dv > 0 else 0
    dd = (nav - nav.cummax()) / nav.cummax()
    calmar = cagr / abs(dd.min()) if dd.min() < 0 else 0
    mo = nav.resample("ME").last().pct_change().dropna()
    return {
        "CAGR": cagr,
        "Vol": vol,
        "Sharpe": sr,
        "Sortino": sortino,
        "Max DD": dd.min(),
        "Calmar": calmar,
        "Monthly Win%": (mo > 0).mean(),
        "Final NAV": nav.iloc[-1],
        "P&L": nav.iloc[-1] - nav.iloc[0],
    }


In [256]:
_ref = max(LEVERAGE_RUNS)
p = perf(ALL_BT[_ref]["nav"])
bt = ALL_BT[_ref]
print("Reference gross leverage:", _ref)
for k, v in p.items():
    if "NAV" in k or "P&L" in k:
        print(f"  {k}: ${v:,.0f}")
    elif "%" in k:
        print(f"  {k}: {v:.1%}")
    else:
        print(f"  {k}: {v:.2%}")
print(f"  Txn Costs: ${bt['cum_costs'].iloc[-1]:,.0f}")
print(f"  Borrow: ${bt['cum_borrow'].iloc[-1]:,.0f}")
print(f"  Margin debit (cum): ${bt['cum_margin_debit'].iloc[-1]:,.0f}")
print(f"  Long P&L (cum): ${bt['cum_long_pnl'].iloc[-1]:,.0f}")
print(f"  Short P&L (cum): ${bt['cum_short_pnl'].iloc[-1]:,.0f}")


Reference gross leverage: 4.75
  CAGR: 38.16%
  Vol: 13.13%
  Sharpe: 290.67%
  Sortino: 505.38%
  Max DD: -6.50%
  Calmar: 586.96%
  Monthly Win%: 79.5%
  Final NAV: $28,843,777
  P&L: $18,896,347
  Txn Costs: $1,141,969
  Borrow: $2,628,349
  Margin debit (cum): $4,340,497
  Long P&L (cum): $87,492,965
  Short P&L (cum): $-60,826,440


## Final export — DC ETF Arb Pairwise Backtest Attribution

Writes `data/backtest/DC ETF Arb Pairwise Backtest Attribution.xlsx` (from repo root) using the backtest outputs (`ALL_PAIR_DAILY`, `ALL_BT`) and the DC template layout (`Daily PnL`, `Monthly Attribution`).

`Daily PnL!PnL` defaults to `book_nav_change_usd` (set `daily_pnl_source="pairwise_daily_net_pnl_usd"` if you want the pair-rollup series instead). **Management** accrues **once per calendar quarter** (posted on the last trading day in that quarter in the backtest index, in **Mar / Jun / Sep / Dec**): `LP_FEE!B` is an **Excel formula** tied to `book_raw` and `dc_pairwise_params` (quarterly cashflow matches `lp_fees_v15.build_lp_fee_daily_cashflow_usd` for the mgmt leg). **Incentive** in `LP_FEE!C` is still **values** from the v15 pass-2 allocation; `P`/`Q` on `Monthly Attribution` roll up `DAILY_CALC` from those columns. `asof_end` (default `2026-03-31`) drops later dates. Underlying borrow in the engine is re-netted by `(date, under)` before export so offsetting `long_sh` in bucket-3 style structures do not double-count. `ALL_PAIRS` leg notionals are formula-driven after export (`long_sh*PU`, `short_sh*ETF`). Fees: `management_fee_rate_annual` / `incentive_fee_rate` / `fee_daycount`.

In [258]:
import importlib
import sys
import time
from pathlib import Path

# Project root = directory that contains scripts/export_dc_etf_arb_pairwise_workbook.py
# (avoids writing to the wrong folder when cwd is repo root vs notebooks/).
_p = Path.cwd().resolve()
_ROOT = None
for _ in range(6):
    if (_p / "scripts" / "export_dc_etf_arb_pairwise_workbook.py").is_file():
        _ROOT = _p
        break
    _p = _p.parent
if _ROOT is None:
    raise FileNotFoundError(
        "Could not find scripts/export_dc_etf_arb_pairwise_workbook.py — open the notebook from the repo or a subfolder and retry."
    )
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import scripts.daily_pair_workbook_formulas as _dpwf
importlib.reload(_dpwf)
import scripts.export_dc_etf_arb_pairwise_workbook as _pair_attr
importlib.reload(_pair_attr)

from scripts.export_dc_etf_arb_pairwise_workbook import export_dc_etf_arb_pairwise_workbook
# Export: book_raw, book_daily, LP_FEE (mgmt last trading day of Q; pass-2 perf). Monthly E=sum(LMB), G=sum(SFB) at EOM in C; col C uses MAX(IF) (no MAXIFS).

_ref = max(LEVERAGE_RUNS)
_base = float(CFG.get("capital_usd", 10_000_000.0))
# Workbook shell: ``template_xlsx=None`` → resolve_template_xlsx() (env DC_ETF_PAIRWISE_TEMPLATE_XLSX,
# repo ``templates/DC_ETF_Arb_Pairwise_Template.xlsx``, then Downloads Pairwise, then 2-sheet fund xlsx).
_golden_layout = Path.home() / "Downloads" / "DC ETF Arb Pairwise Backtest Attribution.xlsx"
_out = (_ROOT / "data" / "backtest" / "DC ETF Arb Pairwise Backtest Attribution.xlsx").resolve()
_t0 = time.time()
_p_written = export_dc_etf_arb_pairwise_workbook(
    ALL_PAIR_DAILY[_ref],
    ALL_BT[_ref],
    _out,
    template_xlsx=None,
    attribution_base_capital=_base,
    daily_pnl_source="book_nav_change_usd",
    management_fee_rate_annual=float(CFG.get("management_fee_rate_annual", 0.02)),
    incentive_fee_rate=float(CFG.get("incentive_fee_rate", 0.20)),
    fee_daycount=float(CFG.get("fee_daycount", 365.0)),
    include_fees_in_daily_pnl=True,
    asof_end="2026-03-31",
    layout_golden_xlsx=_golden_layout if _golden_layout.is_file() else None,
    crystallize_trailing_partial_year=True,
    reallocate_underlying_borrow=True,
)
_st = _p_written.stat()
print("Wrote", _p_written)
print("  size_bytes", _st.st_size, " mtime", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(_st.st_mtime)), f" took {time.time()-_t0:.2f}s")
if _p_written != _out:
    print("  (returned path differs from input; using returned path.)")
# If this prints but Excel still shows old data, close the workbook and reopen (or the file is open elsewhere and export raised PermissionError).

Wrote C:\Users\werdn\Documents\Investing\ls-algo\data\backtest\DC ETF Arb Pairwise Backtest Attribution.xlsx
  size_bytes 16413541  mtime 2026-04-25 22:22:12  took 824.45s
